# COMS4060A/7056A – Assignment 2: Geospatial & Time-Series Analysis of NYC Taxi Trips

**Dataset:** NYC TLC taxi trips (Kaggle *nyc-taxi-trip-duration*), `train.csv` only – 1 458 644 trips, 2016-01-01 → 2016-06-30.
**Deadline:** 28 October 2026 · **Total marks:** 63

This single notebook answers every question; each section header carries the question number.
Reasoning that goes in the NeurIPS-style PDF is repeated here in the markdown cells so the two stay in sync.

| Section | Topic | Marks |
|---|---|---|
| 0 | Setup & data loading | – |
| 1 | Data cleaning | 3 |
| 2 | Feature generation | 3 |
| 3 | Time-based analysis | 14 |
| 4 | Location clusters (heatmaps, DBSCAN hotspots) | 12 |
| 5 | Airports | 9 |
| 6 | Dimensionality reduction (PCA / t-SNE / UMAP) | 22 |

> **Progress:** all sections (0–6) are complete.

## 0. Setup & data loading

All random operations use the fixed seed `SEED = 42` for reproducibility.
The loader looks for `train.csv` in a few places and, if it is absent, reads it straight out of the
`nyc-taxi-trip-duration.zip` that sits in the repository (the zip contains a nested `train.zip`).

In [ ]:
import os, io, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.titleweight": "bold"})

FIG_DIR = "figures"            # every figure used in the PDF is also saved here
os.makedirs(FIG_DIR, exist_ok=True)
def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, name + ".png"), bbox_inches="tight")

# --- constants given in the assignment ---------------------------------------------------
NYC_CENTRE = (40.716662, -74.009899)           # (lat, lon) as given in the brief

In [ ]:
def load_train():
    """Load train.csv from disk, or straight out of the nested zip files in the repo."""
    kw = dict(parse_dates=["pickup_datetime", "dropoff_datetime"])
    for p in ["train.csv", "data/train.csv", "/tmp/claude-0/data/raw/train.csv"]:
        if os.path.exists(p):
            return pd.read_csv(p, **kw)
    with zipfile.ZipFile("nyc-taxi-trip-duration.zip") as outer:
        inner = zipfile.ZipFile(io.BytesIO(outer.read("train.zip")))
        with inner.open("train.csv") as f:
            return pd.read_csv(f, **kw)

raw = load_train()
print(f"Rows: {len(raw):,}   Columns: {raw.shape[1]}")
raw.head()

In [ ]:
print(raw.dtypes, end="\n\n")
print("Missing values per column:\n", raw.isna().sum().to_string(), sep="")
print("\nDuplicate trip ids:", raw["id"].duplicated().sum())
print("Date range:", raw["pickup_datetime"].min(), "→", raw["pickup_datetime"].max())
# does trip_duration agree with dropoff - pickup?
gap = (raw["dropoff_datetime"] - raw["pickup_datetime"]).dt.total_seconds() - raw["trip_duration"]
print("Rows where trip_duration != dropoff - pickup:", int((gap != 0).sum()))
raw.describe().T

**Initial data audit.** There are no missing values and no duplicate ids, and `trip_duration` is exactly
`dropoff_datetime − pickup_datetime` for every row, so there is no timestamp inconsistency to repair.
The `describe()` table already shows the problem the assignment hints at: coordinates as far away as
longitude −121.9 / latitude 51.9 (not New York), passenger counts of 0 and up to 9, a minimum duration of 1 s
and a maximum of 3 526 282 s (≈ 40 days). The following section quantifies and removes these.

---
## 1. Data Cleaning [3 marks]

**Goal:** identify outliers / impossible records and justify their removal.
We examine five aspects: (a) passenger count, (b) geographic location, (c) trip duration, (d) trip distance, (e) average speed.
Distance is the **great-circle (haversine) distance** between pickup and dropoff (formula supplied in the brief);
speed is `distance / duration`. These two derived columns are also required in Question 2.

> Note on haversine: it is a *lower bound* on the road distance actually driven (streets are not straight lines),
> so the speeds we compute under-estimate real speeds. Thresholds are therefore set generously.

In [ ]:
def haversine_np(lon1, lat1, lon2, lat2):
    """Great-circle distance in km between points given in decimal degrees (vectorised)."""
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6367 * c

df = raw.copy()
df["distance_km"] = haversine_np(df.pickup_longitude, df.pickup_latitude,
                                 df.dropoff_longitude, df.dropoff_latitude)
df["speed_kmh"] = df["distance_km"] / (df["trip_duration"] / 3600.0)
df[["trip_duration", "distance_km", "speed_kmh"]].describe(percentiles=[.001, .01, .5, .99, .999]).T

### 1.1 Passenger count

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
vc = raw["passenger_count"].value_counts().sort_index()
ax.bar(vc.index.astype(str), vc.values, color="steelblue")
ax.set_yscale("log"); ax.set_xlabel("passenger_count"); ax.set_ylabel("trips (log scale)")
ax.set_title("Passenger count (raw data)")
for x, v in zip(vc.index.astype(str), vc.values): ax.text(x, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); savefig("q1_passenger_count"); plt.show()

**Reasoning.** Yellow-cab capacity in NYC is 4 passengers (5 for some vehicles; up to 6 for larger vans/minivans, which the data
shows are common: 78 088 trips with 5 and 48 333 with 6 passengers). Counts of **0** (60 trips) have no passenger so
cannot be a real paid trip, and **7, 8, 9** (5 trips in total) exceed the capacity of any taxi. These are data-entry errors
(the field is *driver entered*) so we keep only `1 ≤ passenger_count ≤ 6`.

### 1.2 Geographic outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(raw.pickup_longitude.sample(20000, random_state=SEED), raw.pickup_latitude.sample(20000, random_state=SEED),
                s=2, alpha=.4, color="crimson")
axes[0].set_title("Pickups – raw data (20k sample): world-spanning outliers"); axes[0].set_xlabel("lon"); axes[0].set_ylabel("lat")
h = axes[1].hist2d(raw.pickup_longitude.clip(-74.3, -73.65), raw.pickup_latitude.clip(40.5, 41.0),
                   bins=300, cmap="inferno", norm=plt.matplotlib.colors.LogNorm())
axes[1].set_title("Pickups – zoomed on NYC (clipped to window)"); axes[1].set_xlabel("lon")
plt.tight_layout(); savefig("q1_geo_raw"); plt.show()

In [ ]:
# Bounding box that comfortably contains the five boroughs *and* both airports (JFK -73.78, Newark -74.17)
LON_MIN, LON_MAX = -74.30, -73.65
LAT_MIN, LAT_MAX = 40.49, 40.92
in_box = lambda lon, lat: lon.between(LON_MIN, LON_MAX) & lat.between(LAT_MIN, LAT_MAX)
geo_ok = in_box(df.pickup_longitude, df.pickup_latitude) & in_box(df.dropoff_longitude, df.dropoff_latitude)
print(f"Trips with pickup or dropoff outside the NYC window: {(~geo_ok).sum():,} ({(~geo_ok).mean():.3%})")
print("  of which both ends are outside:",
      int((~in_box(df.pickup_longitude, df.pickup_latitude) & ~in_box(df.dropoff_longitude, df.dropoff_latitude)).sum()))

**Reasoning.** The dataset is *New York City* taxi data, yet the raw coordinates include points near longitude −121.9
(the US west coast), latitude 51.9 (Canada) and −61.3 (the Atlantic). These are GPS errors/zero-fixes, not real trips.
We keep trips whose pickup **and** dropoff both lie in the window lon ∈ [−74.30, −73.65], lat ∈ [40.49, 40.92], which
covers all five boroughs plus Newark (−74.17) and JFK (−73.78) airports (needed for Question 5). A point outside this
window cannot be reached by a taxi inside a few-hours trip, and would dominate every spatial analysis (heatmaps, DBSCAN, PCA scaling).

### 1.3 Trip duration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(np.log10(raw.trip_duration), bins=120, color="steelblue")
axes[0].set_xlabel("log10(trip_duration in s)"); axes[0].set_ylabel("trips"); axes[0].set_title("Duration, log scale (raw)")
for v, l in [(60, "1 min"), (3600, "1 h"), (10800, "3 h"), (86400, "24 h")]:
    axes[0].axvline(np.log10(v), color="crimson", ls="--"); axes[0].text(np.log10(v), axes[0].get_ylim()[1]*.9, l, rotation=90, color="crimson", va="top", ha="right")
axes[1].hist(raw.trip_duration.clip(upper=7200)/60, bins=120, color="steelblue")
axes[1].set_xlabel("minutes (clipped at 120)"); axes[1].set_title("Duration up to 2 h")
axes[2].boxplot(np.log10(raw.trip_duration), vert=False, flierprops=dict(marker=".", markersize=2, alpha=.3))
axes[2].set_xlabel("log10(trip_duration)"); axes[2].set_title("Boxplot (log scale)")
plt.tight_layout(); savefig("q1_duration_raw"); plt.show()

dur = raw.trip_duration
print(f"< 60 s : {(dur<60).sum():,}    < 10 s : {(dur<10).sum():,}")
print(f"> 3 h  : {(dur>3*3600).sum():,}    > 24 h : {(dur>86400).sum():,}   max = {dur.max()/86400:.1f} days")

**Reasoning.** On a log scale the bulk of trips is a smooth, roughly log-normal bump centred at ≈ 11 min, with a thin
extreme tail on **both** sides:
* **Too short (< 60 s).** A cab cannot meaningfully hail, carry and drop a passenger in under a minute; these are
  accidental meter starts, cancelled rides or driver test records (8 595 trips are < 60 s, 1 984 of them < 10 s).
* **Too long (> 3 h).** Intra-city taxi trips essentially never last longer than a few hours; the tail contains trips of
  up to ≈ 40 days (meter left running/never switched off). 2 112 trips exceed 3 h and 4 exceed 24 h; they they distort every mean and
  every duration scaling. The 3 h cap is conservative: it exceeds the longest realistic NYC–airport journeys in heavy traffic.

### 1.4 Trip distance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(np.log10(df.distance_km.clip(lower=1e-3)), bins=120, color="darkorange")
axes[0].set_xlabel("log10(distance in km)"); axes[0].set_title("Haversine distance (raw, log scale)")
axes[0].axvline(np.log10(0.1), color="crimson", ls="--")
axes[1].hist(df.distance_km.clip(upper=40), bins=120, color="darkorange")
axes[1].set_xlabel("km (clipped at 40)"); axes[1].set_title("Haversine distance up to 40 km")
plt.tight_layout(); savefig("q1_distance_raw"); plt.show()
print(f"distance == 0 km: {(df.distance_km == 0).sum():,};  < 100 m: {(df.distance_km < .1).sum():,};  > 100 km: {(df.distance_km > 100).sum():,}")

**Reasoning.** A sizeable spike sits at ≈ 0 km: trips whose pickup and dropoff coordinates are (almost) identical.
Some of these are zero-length because the driver never really drove (cancelled fare) or the GPS failed; for the rest the
speed is meaningless (division by ~0). Trips shorter than **100 m** are below the typical GPS positioning error in
Manhattan's "urban canyon" (≈ 10-50 m), so a 100 m trip is indistinguishable from "didn't move" and cannot represent a
paid taxi fare. On the long side the largest legitimate distance inside our NYC window is below ≈ 50 km
(Newark ↔ the Bronx/Queens is the extreme), so no extra distance upper cap is needed beyond the geographic window and speed filter.

### 1.5 Average speed

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df.speed_kmh.clip(upper=150), bins=150, color="seagreen")
axes[0].axvline(100, color="crimson", ls="--", label="100 km/h cutoff"); axes[0].legend()
axes[0].set_xlabel("average speed km/h (clipped at 150)"); axes[0].set_title("Average speed, raw")
axes[1].scatter(df.sample(30000, random_state=SEED).trip_duration.clip(upper=7200)/60,
                df.sample(30000, random_state=SEED).distance_km.clip(upper=60), s=2, alpha=.3, color="seagreen")
for sp, ls in [(100, "--")]:
    x = np.linspace(0, 120, 100); axes[1].plot(x, sp * x / 60, "crimson", ls=ls, label="100 km/h")
axes[1].set_ylim(0, 60); axes[1].set_xlim(0, 120); axes[1].legend()
axes[1].set_xlabel("duration (min)"); axes[1].set_ylabel("distance (km)"); axes[1].set_title("Distance vs duration (30k sample)")
plt.tight_layout(); savefig("q1_speed_raw"); plt.show()
print(df.speed_kmh.describe(percentiles=[.5, .99, .999]).round(1).to_string())
print(f"> 100 km/h: {(df.speed_kmh > 100).sum():,}   > 200 km/h: {(df.speed_kmh > 200).sum():,}")

**Reasoning.** The speed distribution of genuine trips is a right-skewed bell with median ≈ 13 km/h (heavy Manhattan traffic) and a
long right tail. A tail of absurd values (hundreds to thousands of km/h) exists, produced by trips whose recorded
duration is a few seconds but whose coordinates are kilometres apart. The posted speed limits are 25 mph (40 km/h) in the
city and ≤ 65 mph (105 km/h) on highways; as haversine under-estimates road distance, any trip with an *average*
speed over **100 km/h** is physically implausible. We also remove trips below the minimum realistic speed
(see below) because they are trips that were open for a long time with very little movement (waiting taxis, forgotten meters).

In [ ]:
# very slow trips: show the lower tail of speed for trips that moved at least 100 m
slow = df[(df.distance_km >= .1)]
print((slow.speed_kmh < 1).sum(), "trips with average speed < 1 km/h  (walking pace is ~5 km/h)")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(slow.speed_kmh.clip(upper=10), bins=100, color="seagreen"); ax.axvline(1, color="crimson", ls="--")
ax.set_xlabel("speed km/h (0-10 shown)"); ax.set_title("Lower tail of average speed"); plt.tight_layout(); savefig("q1_speed_low"); plt.show()

Trips averaging **< 1 km/h** are slower than walking (~5 km/h) over the entire trip: the vehicle was effectively
stationary with the meter running, so these are not representative trips and are removed.

### 1.6 Applying the filters – cleaning funnel

In [ ]:
rules = {
    "passenger_count not in [1, 6]":             ~df.passenger_count.between(1, 6),
    "coordinates outside NYC window":            ~geo_ok,
    "duration < 60 s":                           df.trip_duration < 60,
    "duration > 3 h":                            df.trip_duration > 3 * 3600,
    "distance < 100 m":                          df.distance_km < 0.1,
    "average speed > 100 km/h":                  df.speed_kmh > 100,
    "average speed < 1 km/h":                    df.speed_kmh < 1,
}
rows, remaining = [], pd.Series(True, index=df.index)
for name, bad in rules.items():
    newly = bad & remaining
    rows.append({"rule": name, "flagged_alone": int(bad.sum()), "newly_removed_in_order": int(newly.sum())})
    remaining &= ~bad
funnel = pd.DataFrame(rows)
funnel["flagged_alone_%"] = (funnel.flagged_alone / len(df) * 100).round(3)
print(f"Original: {len(df):,}   Kept: {remaining.sum():,}   Removed: {(~remaining).sum():,} ({(~remaining).mean():.2%})")
funnel

In [ ]:
clean = df[remaining].reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(clean.trip_duration / 60, bins=120, color="steelblue"); axes[0].set_xlabel("minutes"); axes[0].set_title("Duration (clean)")
axes[1].hist(clean.distance_km, bins=120, color="darkorange"); axes[1].set_xlabel("km"); axes[1].set_title("Distance (clean)")
axes[2].hist(clean.speed_kmh, bins=120, color="seagreen"); axes[2].set_xlabel("km/h"); axes[2].set_title("Average speed (clean)")
plt.tight_layout(); savefig("q1_clean_distributions"); plt.show()

clean[["passenger_count", "trip_duration", "distance_km", "speed_kmh"]].describe(percentiles=[.01, .5, .99]).T.round(2)

### Summary of Question 1

| Rule | Justification |
|---|---|
| `passenger_count ∉ [1, 6]` | 0 passengers = no fare; 7–9 exceed any taxi's capacity → data-entry errors |
| pickup/dropoff outside lon [−74.30, −73.65], lat [40.49, 40.92] | not in NYC (coordinates in Atlantic, Canada, west coast); window still includes JFK and Newark |
| duration < 60 s | cancelled/accidental meter start, no real journey |
| duration > 3 h | meter left running; up to ≈ 40 days; unrealistic intra-city trip |
| distance < 100 m | below GPS accuracy – trip did not go anywhere; speed undefined |
| speed > 100 km/h | physically impossible for the *average* of a city trip given haversine under-estimates distance |
| speed < 1 km/h | slower than walking over the whole trip → stationary vehicle with meter running |

The filters remove only about 1.4 % of the data (20 448 of 1 458 644 trips) (see funnel), so the cleaned data set is still essentially the full
sample, while mean-based statistics, scalers and distance-based methods (DBSCAN, t-SNE, UMAP) are no longer dominated by errors.
The table above shows how many trips each rule flagged on its own and how many it removed in sequence (rules overlap).

---
## 2. Feature generation [3 marks]

Required: **trip distance**, **day of week**, **average speed**. We add further features that are used in later sections.
Features are built on the *cleaned* data (the order is irrelevant to the result; distance and speed were already
computed above to drive the cleaning, and are re-derived here from scratch so this section stands alone).

In [ ]:
feat = clean.copy()

# ---- required features --------------------------------------------------------------------
feat["distance_km"] = haversine_np(feat.pickup_longitude, feat.pickup_latitude,
                                   feat.dropoff_longitude, feat.dropoff_latitude)          # 1. distance of trip
feat["speed_kmh"] = feat["distance_km"] / (feat["trip_duration"] / 3600.0)                  # 3. average speed of trip
feat["day_of_week"] = feat.pickup_datetime.dt.dayofweek                                     # 2. day of week (0=Mon … 6=Sun)
feat["day_name"] = pd.Categorical(feat.pickup_datetime.dt.day_name(),
                                  categories=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
                                  ordered=True)

# ---- extra features -------------------------------------------------------------------------
feat["pickup_hour"] = feat.pickup_datetime.dt.hour
feat["pickup_date"] = feat.pickup_datetime.dt.normalize()
feat["pickup_month"] = feat.pickup_datetime.dt.month
feat["is_weekend"] = feat.day_of_week >= 5
feat["duration_min"] = feat.trip_duration / 60.0
feat["hour_sin"] = np.sin(2 * np.pi * feat.pickup_hour / 24)      # cyclic hour encoding, used in Q6
feat["hour_cos"] = np.cos(2 * np.pi * feat.pickup_hour / 24)
# Manhattan-style (L1) distance: grid streets mean the real route is closer to |dlat|+|dlon| than to the straight line
feat["manhattan_km"] = (haversine_np(feat.pickup_longitude, feat.pickup_latitude, feat.dropoff_longitude, feat.pickup_latitude)
                      + haversine_np(feat.dropoff_longitude, feat.pickup_latitude, feat.dropoff_longitude, feat.dropoff_latitude))
# initial compass bearing (degrees, 0=north): captures the dominant direction of travel on the Manhattan grid
dlon = np.radians(feat.dropoff_longitude - feat.pickup_longitude)
la1, la2 = np.radians(feat.pickup_latitude), np.radians(feat.dropoff_latitude)
feat["bearing_deg"] = (np.degrees(np.arctan2(np.sin(dlon) * np.cos(la2),
                        np.cos(la1) * np.sin(la2) - np.sin(la1) * np.cos(la2) * np.cos(dlon))) + 360) % 360
# distances to the city centre given in the brief
feat["pickup_dist_centre_km"] = haversine_np(feat.pickup_longitude, feat.pickup_latitude, NYC_CENTRE[1], NYC_CENTRE[0])
feat["dropoff_dist_centre_km"] = haversine_np(feat.dropoff_longitude, feat.dropoff_latitude, NYC_CENTRE[1], NYC_CENTRE[0])

new_cols = ["distance_km", "speed_kmh", "day_of_week", "day_name", "pickup_hour", "pickup_date", "pickup_month", "is_weekend",
            "duration_min", "hour_sin", "hour_cos", "manhattan_km", "bearing_deg", "pickup_dist_centre_km", "dropoff_dist_centre_km"]
print(f"Clean trips: {len(feat):,}   |   new feature columns: {len(new_cols)}")
feat[["pickup_datetime", "trip_duration"] + new_cols].head()

In [ ]:
feat[["distance_km", "speed_kmh", "duration_min", "manhattan_km", "pickup_dist_centre_km", "dropoff_dist_centre_km"]].describe().T.round(2)

In [ ]:
# Sanity checks on the generated features
assert feat.day_of_week.between(0, 6).all() and feat.pickup_hour.between(0, 23).all()
assert (feat.distance_km > 0).all() and (feat.speed_kmh > 0).all()
assert (feat.manhattan_km >= feat.distance_km - 1e-9).all()     # L1 route can never be shorter than the straight line
assert np.allclose(feat.hour_sin**2 + feat.hour_cos**2, 1)
print("Median Manhattan/straight-line ratio:", round((feat.manhattan_km / feat.distance_km).median(), 3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(feat.distance_km.clip(upper=25), bins=100, color="darkorange"); axes[0].set_title("Distance of trip (km)")
feat.day_name.value_counts().sort_index().plot.bar(ax=axes[1], color="slateblue"); axes[1].set_title("Trips per day of week"); axes[1].set_xlabel("")
axes[2].hist(feat.speed_kmh, bins=100, color="seagreen"); axes[2].set_title("Average speed (km/h)")
plt.tight_layout(); savefig("q2_features_overview"); plt.show()

### Summary of Question 2

| Column | Definition | Why it is useful |
|---|---|---|
| `distance_km` **(required)** | haversine distance pickup→dropoff (km) | trip length; used in speed, Q4–Q6 |
| `day_of_week` / `day_name` **(required)** | 0 = Monday … 6 = Sunday, from the pickup time | Q3 weekly patterns |
| `speed_kmh` **(required)** | `distance_km / (trip_duration / 3600)` | Q3.5 speed vs hour; Q6 |
| `pickup_hour`, `pickup_date`, `pickup_month` | calendar components of the pickup time | Q3 time analysis, holiday lookup |
| `is_weekend` | day_of_week ≥ 5 | Q3.3, Q4.1 |
| `duration_min` | `trip_duration` in minutes | readability |
| `hour_sin`, `hour_cos` | cyclic encoding of the hour | Q6 (23:00 is close to 00:00) |
| `manhattan_km` | L1 route distance along the grid | better proxy for road distance than haversine |
| `bearing_deg` | initial compass bearing | direction of travel |
| `pickup_dist_centre_km`, `dropoff_dist_centre_km` | distance to (40.716662, −74.009899) | how central a trip is |

All checks above pass (valid ranges, Manhattan ≥ straight-line, unit-circle hour encoding). The working data frame for the
rest of the notebook is **`feat`**.

---
## 3. Time-based analysis [14 marks]

All questions use the **pickup time** unless stated otherwise. The data cover 182 calendar days (2016-01-01 → 2016-06-30 = 26 full weeks),
so each day of the week occurs exactly 26 times, which makes raw weekday totals directly comparable.

Two things are prepared once for the whole section:
* `DAYS` – ordered weekday names, and a per-day-and-hour count table `hourly` (rows = calendar date, columns = hour 0–23).
* **Winter Storm Jonas (23–24 Jan 2016).** A record blizzard hit NYC on Sat 23 Jan with a travel ban in place. The data show it clearly
  (see 3.1: only 1 614 trips on the 23rd versus ≈ 8 400 on a typical Saturday). Because it is a one-off weather event unrelated to the
  weekly rhythm we are studying, it is *flagged* and excluded when computing "typical" baselines (23–25 Jan; the 25th is still recovering).

In [ ]:
from scipy import stats

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
daily = feat.groupby("pickup_date").size().rename("trips").to_frame()
daily["dow"] = daily.index.dayofweek
daily["day_name"] = pd.Categorical([DAYS[d] for d in daily.dow], categories=DAYS, ordered=True)
daily["month"] = daily.index.month

STORM = pd.to_datetime(["2016-01-23", "2016-01-24", "2016-01-25"])           # Winter Storm Jonas and its aftermath
daily["storm"] = daily.index.isin(STORM)

hourly = feat.groupby(["pickup_date", "pickup_hour"]).size().unstack(fill_value=0)   # date x hour counts
print("Calendar days:", len(daily), "| occurrences of each weekday:", daily.dow.value_counts().sort_index().unique())
daily.sort_values("trips").head(5)

### 3.1 Which day of the week is the most popular? [2 marks]

In [ ]:
total_by_day = feat.groupby("day_name", observed=True).size()
typical = daily[~daily.storm]                                                  # daily counts without the blizzard days
mean_by_day = typical.groupby("day_name", observed=True).trips.agg(["mean", "std", "count"])

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
cols = ["#4c72b0"] * 7; cols[total_by_day.values.argmax()] = "#c44e52"
axes[0].bar(DAYS, total_by_day.values / 1e3, color=cols)
for i, v in enumerate(total_by_day.values): axes[0].text(i, v / 1e3, f"{v/1e3:.0f}k", ha="center", va="bottom", fontsize=9)
axes[0].set_ylim(150, 230); axes[0].set_ylabel("total trips (thousands)"); axes[0].set_title("Total pickups per weekday (Jan–Jun)")
axes[0].tick_params(axis="x", rotation=45)

sns.boxplot(data=daily, x="day_name", y="trips", ax=axes[1], color="#9ecae1", fliersize=3)
sns.stripplot(data=daily, x="day_name", y="trips", ax=axes[1], color="k", size=3, alpha=.5)
axes[1].set_title("Distribution of daily trip counts (each dot = one date)"); axes[1].set_xlabel(""); axes[1].set_ylabel("trips per day")
axes[1].tick_params(axis="x", rotation=45)
axes[1].annotate("Blizzard\n23 Jan", xy=(5, 1614), xytext=(5.1, 3500), fontsize=8, arrowprops=dict(arrowstyle="->"))

piv = daily.pivot_table(index="month", columns="day_name", values="trips", aggfunc="mean", observed=True)
sns.heatmap(piv, annot=True, fmt=".0f", cmap="YlOrRd", ax=axes[2], cbar_kws={"label": "mean trips / day"})
axes[2].set_title("Mean daily trips by month × weekday"); axes[2].set_ylabel("month (1 = Jan)")
plt.tight_layout(); savefig("q3_1_popular_day"); plt.show()

print(mean_by_day.round(0).to_string())
# Is the winner statistically distinguishable from the runner-up? (Welch t-test on daily counts, storm excluded)
top2 = mean_by_day["mean"].nlargest(2).index
a, b = (typical[typical.day_name == d].trips for d in top2)
t, p = stats.ttest_ind(a, b, equal_var=False)
print(f"\nTop two by mean daily trips: {top2[0]} ({a.mean():.0f}) vs {top2[1]} ({b.mean():.0f}); Welch t = {t:.2f}, p = {p:.3f}")
print("Friday in the top-3 busiest weekday of the month in", int((piv.rank(axis=1, ascending=False)["Friday"] <= 3).sum()), "of 6 months")

**Answer – Friday and Saturday are the two most popular days, and they are statistically tied; both clearly beat Sunday–Wednesday.**

* **Totals over the six months:** Friday is highest (220 378 pickups), then Saturday (217 740), Thursday (215 509), Wednesday (207 384),
  Tuesday (199 969), Sunday (192 396) and Monday (184 820, the quietest). Every weekday occurs 26 times, so totals are directly comparable
  (Friday vs Monday: +19 %).
* **Typical days (blizzard excluded):** removing Winter Storm Jonas (which fell on a Saturday/Sunday and cost Saturday ≈ 7 000 trips) puts **Saturday first** (≈ 8 645 trips/day) and Friday second (≈ 8 476).
  The Welch t-test on daily counts gives p = 0.37, so the Friday–Saturday difference is **not significant**: the honest answer is a tie, with Friday winning on raw totals only because of the blizzard.
  Friday is among the three busiest weekdays in all 6 months.
* **Robustness:** the box-plots show Friday's and Saturday's distributions of daily counts sit above Sunday–Wednesday (Saturday also has the largest spread).
  The month × weekday heat-map lets you check that the ordering is not driven by a single month.
* **Why:** Thursday–Saturday are the "going-out" days (dinner, bars, shows, then the ride home) on top of the regular commute; Monday and Sunday are low because there is neither a
  full leisure demand (Sunday) nor a strong start-of-week demand (Monday). The single lowest day in the data set is the blizzard itself (Sat 23 Jan, 1 614 trips) – bad weather suppresses demand far more than any weekday effect.

### 3.2 What hour of the day is the most popular on each day? [3 marks]

In [ ]:
# average number of trips in each (weekday, hour) slot, per calendar occurrence of that weekday (storm days excluded)
hourly_t = hourly[~hourly.index.isin(STORM)]
dow_idx = hourly_t.index.dayofweek
slot = hourly_t.groupby(dow_idx).mean()                  # 7 x 24: mean trips in that hour on that weekday
slot.index = DAYS
peak = pd.DataFrame({"peak hour": slot.idxmax(axis=1),
                     "mean trips in peak hour": slot.max(axis=1).round(0),
                     "quietest hour": slot.idxmin(axis=1),
                     "mean trips in quietest hour": slot.min(axis=1).round(0)})
print(peak.to_string())

fig = plt.figure(figsize=(18, 9))
gs = fig.add_gridspec(2, 4, height_ratios=[1, 1])
ax = fig.add_subplot(gs[0, :2])
sns.heatmap(slot, cmap="YlOrRd", ax=ax, cbar_kws={"label": "mean trips"}, linewidths=.2)
ax.set_title("Mean pickups per hour, by weekday (storm excluded)"); ax.set_xlabel("pickup hour"); ax.set_ylabel("")
ax = fig.add_subplot(gs[0, 2:])
for d, c in zip(DAYS, sns.color_palette("husl", 7)):
    ax.plot(slot.columns, slot.loc[d], label=d, color=c, lw=2, ls="-" if d not in ("Saturday", "Sunday") else "--")
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("pickup hour"); ax.set_ylabel("mean trips"); ax.legend(ncol=2, fontsize=8)
ax.set_title("Hourly distribution – all days overlaid")
plt.tight_layout(); savefig("q3_2_hour_heatmap_lines"); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey=True)
for a, d in zip(axes.ravel(), DAYS):
    v = slot.loc[d]
    a.bar(v.index, v.values, color=["#c44e52" if h == v.idxmax() else "#4c72b0" for h in v.index])
    a.set_title(f"{d}  (peak {v.idxmax():02d}:00)"); a.set_xticks(range(0, 24, 3))
for a in axes[:, 0]: a.set_ylabel("mean trips")
axes.ravel()[-1].axis("off")
plt.suptitle("Distribution of pickups over the hours of the day, one panel per weekday", y=1.02, fontweight="bold")
plt.tight_layout(); savefig("q3_2_hour_distribution_per_day"); plt.show()

**Observations** (peak hours from the table printed above):

| Day | Most popular hour | Quietest hour |
|---|---|---|
| Mon / Tue | 18:00 | 03:00 |
| Wed / Fri | 19:00 | 03:00 / 04:00 |
| Thu | 21:00 | 04:00 |
| Sat | 23:00 | 05:00 |
| Sun | 00:00 | 05:00 |

* **Weekdays (Mon–Fri)** share one shape: an almost empty night (minimum at 03:00–04:00), a steep rise at **06:00–08:00** (morning commute), a roughly flat plateau through the working day,
  and the maximum in the **evening, 18:00–21:00** (evening commute plus dinner and going out; on Thursday it is pushed later, to 21:00, as the week's going-out night begins).
* **Saturday and Sunday** have no morning commuter peak. Demand rises slowly through the late morning to a broad afternoon plateau, and the maximum is at **23:00 on Saturday and 00:00 on Sunday** –
  i.e. Friday night and Saturday night spilling past midnight into the next calendar day. Nights are also far busier than on weekdays: the quietest hour on Friday/Saturday/Sunday still has 76–87 pickups against 38–52 on Monday–Thursday.
* **Possible reasons:** (i) commuting (office hours, peaks around 08:00 and 18:00); (ii) dinner, theatre and bar traffic from 19:00 to midnight, with 4 a.m. bar closing time driving weekend rides home between 01:00 and 04:00;
  (iii) the dip at ≈ 16:00 followed by a jump at 17:00 on weekdays is consistent with the well-known **taxi shift change** (day drivers hand over around 16:00–17:00, so fewer meters start); (iv) between 03:00 and 05:00 almost nobody travels except airport runs and shift workers, hence the global minimum.

### 3.3 Weekdays vs weekends [2 marks]

In [ ]:
wk = hourly_t.groupby(hourly_t.index.dayofweek >= 5).mean().T
wk.columns = ["Weekday (Mon–Fri)", "Weekend (Sat–Sun)"]
clean_nostorm = feat[~feat.pickup_date.isin(STORM)]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].plot(wk.index, wk.iloc[:, 0], "-o", label=wk.columns[0]); axes[0].plot(wk.index, wk.iloc[:, 1], "-s", label=wk.columns[1])
axes[0].set_xticks(range(0, 24, 2)); axes[0].set_xlabel("pickup hour"); axes[0].set_ylabel("mean trips per day in that hour")
axes[0].set_title("Average daily profile"); axes[0].legend()
axes[0].fill_between(wk.index, wk.iloc[:, 0], wk.iloc[:, 1], where=wk.iloc[:, 1] > wk.iloc[:, 0], alpha=.15, color="C1")
axes[0].fill_between(wk.index, wk.iloc[:, 0], wk.iloc[:, 1], where=wk.iloc[:, 1] <= wk.iloc[:, 0], alpha=.15, color="C0")

share = wk / wk.sum()
axes[1].bar(share.index - .2, share.iloc[:, 0] * 100, .4, label="Weekday"); axes[1].bar(share.index + .2, share.iloc[:, 1] * 100, .4, label="Weekend")
axes[1].set_xlabel("pickup hour"); axes[1].set_ylabel("% of the day's trips"); axes[1].set_title("Share of daily trips by hour"); axes[1].legend()

comp = clean_nostorm.groupby("is_weekend").agg(distance_km=("distance_km", "mean"), duration_min=("duration_min", "mean"),
        speed_kmh=("speed_kmh", "mean"), passengers=("passenger_count", "mean"), pickup_dist_centre_km=("pickup_dist_centre_km", "mean"))
comp.index = ["Weekday", "Weekend"]
(comp / comp.loc["Weekday"]).T.plot.barh(ax=axes[2]); axes[2].axvline(1, color="k", lw=.8)
axes[2].set_title("Trip characteristics relative to weekday (=1)"); axes[2].set_xlabel("ratio")
plt.tight_layout(); savefig("q3_3_weekday_vs_weekend"); plt.show()

print("Mean trips per day: weekday = %.0f, weekend = %.0f" % (daily[~daily.storm & (daily.dow < 5)].trips.mean(), daily[~daily.storm & (daily.dow >= 5)].trips.mean()))
print(comp.round(2).to_string())
print("\nShare of trips between 00:00-05:00: weekday %.1f%%, weekend %.1f%%" % tuple(share.iloc[0:5].sum() * 100))
print("Share of trips between 07:00-10:00: weekday %.1f%%, weekend %.1f%%" % tuple(share.iloc[7:10].sum() * 100))
print("Share of trips between 18:00-22:00: weekday %.1f%%, weekend %.1f%%" % tuple(share.iloc[18:22].sum() * 100))

**Differences between weekdays and weekends** (storm excluded):

1. **Total volume is similar** (≈ 7 900 trips on a weekday vs ≈ 8 100 on a weekend day), but it is *distributed differently* over the day.
2. **Weekday shape = commuter-driven:** a sharp 07:00–09:00 peak (15.3 % of weekday trips between 07:00 and 10:00, only 7.4 % at weekends) and a larger 18:00–22:00 evening share (25.1 % vs 21.0 %).
3. **Weekend shape = leisure-driven:** no morning rush; demand builds slowly from late morning to a broad plateau of
   ≈ 13:00–19:00 (shopping, brunch, tourism, events); the share of trips between 00:00 and 05:00 is ≈ 2.8× higher on weekends (19.8 % vs 7.0 %; these are the Friday- and Saturday-night
   outings returning home), and the weekend curve is also above the weekday curve from roughly 11:00 to 17:00.
4. **Trip characteristics:** weekend trips are **faster** (≈ 15.9 vs 13.9 km/h) because the streets are less congested,
   **shorter in time** (≈ 12.9 vs 14.4 min) for a similar distance (3.5 vs 3.4 km), carry slightly **more passengers** (1.72 vs 1.64 – groups/families rather than solo business travellers)
   and start slightly closer to the centre (5.3 vs 5.6 km). The weekend speed advantage holds between 05:00 and 19:00; at night (00:00–04:00 and 20:00–23:00) weekend streets are no faster, because they are busy with nightlife traffic.
5. **What accounts for it:** the working week – commute, congestion and business trips – versus a social calendar of nightlife, brunch, tourism and events.

### 3.4 Holidays [5 marks]

2016 dates (the correct dates for the relevant year; all fall inside Jan–Jun 2016):

| Holiday | 2016 date | Weekday |
|---|---|---|
| Martin Luther King Jr. Day (3rd Monday of Jan) | 18 Jan | Monday |
| Valentine's Day | 14 Feb | Sunday |
| St Patrick's Day | 17 Mar | Thursday |
| Easter Sunday | 27 Mar | Sunday |
| Memorial Day (last Monday of May) | 30 May | Monday |

**Method:** a holiday cannot be judged against the average day – it must be compared with *ordinary days of the same weekday*.
For each holiday we build a baseline of all other same-weekday dates (excluding the other holidays and the blizzard days), then compare (a) the daily total,
as a ratio and a z-score against the baseline's day-to-day variation, (b) the hour-by-hour profile, and (c) speed and distance.

In [ ]:
HOLIDAYS = {"Martin Luther King Day": "2016-01-18", "Valentine's Day": "2016-02-14", "St Patrick's Day": "2016-03-17",
            "Easter Sunday": "2016-03-27", "Memorial Day": "2016-05-30"}
HOLIDAYS = {k: pd.Timestamp(v) for k, v in HOLIDAYS.items()}
EXCLUDE = set(HOLIDAYS.values()) | set(STORM)
dow_all = feat.pickup_datetime.dt.dayofweek

rows = []
fig_data = {}
for name, d in HOLIDAYS.items():
    dw = d.dayofweek
    base_days = daily[(daily.dow == dw) & (~daily.index.isin(EXCLUDE))]
    base_h = hourly[(hourly.index.dayofweek == dw) & (~hourly.index.isin(EXCLUDE))]
    h_trips = feat[feat.pickup_date == d]
    b_trips = feat[(dow_all == dw) & (~feat.pickup_date.isin(EXCLUDE))]
    rows.append({"holiday": name, "date": d.date(), "weekday": d.day_name(),
                 "trips": int(daily.loc[d, "trips"]), "baseline_mean": round(base_days.trips.mean()),
                 "ratio": round(daily.loc[d, "trips"] / base_days.trips.mean(), 3),
                 "z_score": round((daily.loc[d, "trips"] - base_days.trips.mean()) / base_days.trips.std(), 2),
                 "peak_hour": int(hourly.loc[d].idxmax()), "baseline_peak_hour": int(base_h.mean().idxmax()),
                 "speed_kmh": round(h_trips.speed_kmh.mean(), 1), "baseline_speed": round(b_trips.speed_kmh.mean(), 1),
                 "dist_km": round(h_trips.distance_km.mean(), 2), "baseline_dist": round(b_trips.distance_km.mean(), 2),
                 "n_baseline_days": len(base_days)})
    fig_data[name] = (base_h, hourly.loc[d])
hol_tbl = pd.DataFrame(rows).set_index("holiday")
hol_tbl

In [ ]:
# (a) the whole six months with holidays marked
fig, ax = plt.subplots(figsize=(16, 4.5))
ax.plot(daily.index, daily.trips, color="#4c72b0", lw=1, label="daily trips")
ax.plot(daily.index, daily.trips.rolling(7, center=True).mean(), color="k", lw=1.5, label="7-day mean")
for (name, d), c in zip(HOLIDAYS.items(), sns.color_palette("Set1", 5)):
    ax.axvline(d, color=c, ls="--"); ax.annotate(name, (d, daily.trips.max() * 1.0), rotation=90, color=c, va="top", ha="right", fontsize=9)
ax.axvspan(STORM[0], STORM[-1], color="grey", alpha=.25, label="Winter Storm Jonas (excluded from baselines)")
ax.set_ylabel("trips per day"); ax.set_title("Daily trips, Jan–Jun 2016, with holidays"); ax.legend(loc="lower right")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b")); plt.tight_layout(); savefig("q3_4_timeline_holidays"); plt.show()

# (b) daily total versus the same-weekday baseline
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8), sharey=False)
for a, (name, d) in zip(axes, HOLIDAYS.items()):
    b = daily[(daily.dow == d.dayofweek) & (~daily.index.isin(EXCLUDE))].trips
    a.hist(b, bins=10, color="#9ecae1", edgecolor="w"); a.axvline(daily.loc[d, "trips"], color="crimson", lw=2.5, label="holiday")
    a.axvline(b.mean(), color="k", ls="--", label="baseline mean")
    a.set_title(f"{name}\n(z = {hol_tbl.loc[name, 'z_score']:+.1f})", fontsize=10); a.set_xlabel(f"trips on a {d.day_name()}")
axes[0].legend(fontsize=8); axes[0].set_ylabel("# of baseline days")
plt.tight_layout(); savefig("q3_4_holiday_totals"); plt.show()

# (c) hourly profile of each holiday versus the same-weekday baseline (mean ± 1 sd)
fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharex=True)
for a, (name, (base_h, hol)) in zip(axes.ravel(), fig_data.items()):
    m, s = base_h.mean(), base_h.std()
    a.fill_between(m.index, m - s, m + s, color="#9ecae1", alpha=.6, label="same weekday: mean ± 1 sd")
    a.plot(m.index, m, color="#08519c", lw=1.5); a.plot(hol.index, hol.values, color="crimson", lw=2.5, marker="o", ms=4, label=name)
    a.set_title(f"{name} – {HOLIDAYS[name].strftime('%a %d %b')}"); a.set_xticks(range(0, 24, 3)); a.legend(fontsize=8); a.set_ylabel("trips / hour")
ax = axes.ravel()[-1]
for (name, (base_h, hol)), c in zip(fig_data.items(), sns.color_palette("Set1", 5)):
    ax.plot(hol.index, hol.values / base_h.mean().values, color=c, label=name, lw=1.8)
ax.axhline(1, color="k", lw=.8); ax.set_title("Holiday ÷ same-weekday baseline, by hour"); ax.set_ylim(0, 3); ax.legend(fontsize=7); ax.set_xlabel("pickup hour")
plt.tight_layout(); savefig("q3_4_holiday_hourly"); plt.show()

**Do patterns change on holidays?** (see the table, the totals histograms and the hourly overlays above)

* **Memorial Day (Mon 30 May)** is the only holiday with a clear, statistically unusual change in the daily total: 5 491 trips, only
  **≈ 76 %** of a normal Monday and about −5.4 standard deviations from the baseline. The hourly plot shows the morning commute almost vanishing (07:00 at only ≈ 24 % of a normal Monday),
  the peak hour moves from 18:00 to 16:00, and after-midnight trips are much higher than normal (≈ 2.5× at 02:00 – the tail of Sunday night's long-weekend outings).
  Trips are **faster and longer** (19.1 vs 15.0 km/h and 4.3 vs 3.55 km): the roads are empty because the offices are closed, and a long-weekend departure from the city (including airport trips) is a plausible reason for the longer distances.
* **Martin Luther King Day (Mon 18 Jan)** is a federal holiday, but the total is within normal range (≈ 97 % of a Monday, z ≈ −0.6). The shape does change: the morning commute is
  much weaker (≈ 45 % of normal at 07:00), speed is higher (16.8 vs 15.0 km/h), and the small hours are busier (≈ 1.6× at 02:00), consistent with a long weekend in which people go out on Sunday night.
* **Valentine's Day (Sun 14 Feb)** has the highest total relative to its baseline (≈ +10.6 % on an ordinary Sunday), although at z ≈ +1.4 this is a modest rather than decisive increase. The shape changes more than the volume:
  the peak hour is 17:00 instead of the small hours (00:00) of an ordinary Sunday, i.e. demand shifts into the early evening (dinner reservations) and the after-midnight spill-over from Saturday night is weaker (≈ 0.72× at 01:00);
  late evening is higher (≈ 1.4× at 23:00). Trips are shorter (3.3 vs 3.8 km) and slower (15.8 vs 16.8 km/h): short hops in busy central areas.
* **St Patrick's Day (Thu 17 Mar)** has essentially the normal total (98 % of a Thursday, z ≈ −0.4) and the same overall shape; the evening peak is an hour earlier (20:00 vs 21:00). The largest deviations are a dip
  at midday (≈ 0.81× at 12:00, plausibly from the 5th Avenue parade closing streets in Midtown) and slightly higher late-evening demand (≈ 1.09× at 23:00). Speed (13.9 vs 13.5 km/h) and distance (3.5 vs 3.4 km) are effectively unchanged.
* **Easter Sunday (27 Mar)** is the closest to a normal day: 96 % of a normal Sunday (z ≈ −0.5), same peak hour (00:00), only slightly higher speed (17.5 vs 16.8 km/h); the largest hourly deviations are
  ≈ 0.76× at 05:00 and ≈ 1.16× at 23:00. Easter is a family/religious day with no large city-wide event, and taxi use is hardly affected.

**Overall:** only Memorial Day produces a daily total outside the usual day-to-day variation; for the other four the *total* is within about 1.5 standard deviations of an ordinary same-weekday, but the
*hourly shape* shifts (weaker commute peaks on the two Monday holidays, a changed evening peak on Valentine's Day). Holiday effects are therefore mostly **redistributions of demand across hours** rather than changes in volume —
with the exception of the long-weekend Monday where volume truly falls. (Caveat: we have a single occurrence of each holiday, so the "baseline" comparison is the best available but a one-off event such as weather could still play a role.)

### 3.5 How does average speed change through the day? [2 marks]

In [ ]:
sp = clean_nostorm.groupby("pickup_hour").speed_kmh.describe(percentiles=[.25, .5, .75])
sp_wk = clean_nostorm.groupby(["is_weekend", "pickup_hour"]).speed_kmh.mean().unstack(0); sp_wk.columns = ["Weekday", "Weekend"]
dist_h = clean_nostorm.groupby("pickup_hour").distance_km.mean()
sp_dh = clean_nostorm.pivot_table(index="day_name", columns="pickup_hour", values="speed_kmh", aggfunc="mean", observed=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
ax = axes[0]
ax.fill_between(sp.index, sp["25%"], sp["75%"], color="#9ecae1", alpha=.6, label="IQR")
ax.plot(sp.index, sp["mean"], "-o", color="#08519c", label="mean"); ax.plot(sp.index, sp["50%"], "--", color="k", label="median")
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("pickup hour"); ax.set_ylabel("average trip speed (km/h)"); ax.legend()
ax.axvline(sp["mean"].idxmax(), color="crimson", ls=":"); ax.set_title(f"Speed through the day (fastest: {sp['mean'].idxmax():02d}:00)")
ax = axes[1]
ax.plot(sp_wk.index, sp_wk.Weekday, "-o", label="Weekday"); ax.plot(sp_wk.index, sp_wk.Weekend, "-s", label="Weekend")
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("pickup hour"); ax.set_ylabel("mean speed (km/h)"); ax.legend(); ax.set_title("Weekday vs weekend")
ax2 = ax.twinx(); ax2.bar(dist_h.index, dist_h.values, alpha=.15, color="grey"); ax2.set_ylabel("mean trip distance (km, grey bars)"); ax2.grid(False)
sns.heatmap(sp_dh, cmap="RdYlGn", ax=axes[2], cbar_kws={"label": "km/h"}); axes[2].set_title("Mean speed: weekday × hour"); axes[2].set_ylabel("")
plt.tight_layout(); savefig("q3_5_speed_by_hour"); plt.show()

print("Fastest hour:", int(sp["mean"].idxmax()), "→", round(sp["mean"].max(), 1), "km/h;   slowest hour:", int(sp["mean"].idxmin()), "→", round(sp["mean"].min(), 1), "km/h")
print("Mean speed 08:00-18:00 (working day):", round(clean_nostorm[clean_nostorm.pickup_hour.between(8, 18)].speed_kmh.mean(), 1),
      "km/h;  00:00-05:00:", round(clean_nostorm[clean_nostorm.pickup_hour <= 5].speed_kmh.mean(), 1), "km/h")
print("Mean distance at 05:00 / 15:00:", round(dist_h[5], 2), "/", round(dist_h[15], 2), "km")
sp[["mean", "50%"]].round(1).T

**Answer – trips are fastest in the early morning, around 05:00 (≈ 24–25 km/h on average),** and slowest in the middle of the working day.

* **Night (00:00–05:00):** speed climbs steadily from ≈ 18 km/h at midnight to the daily maximum of ≈ 24.6 km/h at 05:00 – the streets are nearly empty and traffic lights are the only constraint. The 05:00 peak is also partly a
  *trip-mix* effect: at that hour the mean trip is much longer (airport runs and highway/bridge trips), and longer trips spend more time at highway speed, so their average speed is higher.
* **Sharp drop at 06:00–08:00:** as the morning rush begins, speed falls from ≈ 24.6 km/h (05:00) to ≈ 13 km/h at 08:00 – the biggest change in the day.
* **Daytime (08:00–18:00):** a flat, congested plateau of ≈ 12–13 km/h (slowest around 12:00–15:00 at ≈ 12.1–12.3 km/h): Manhattan's grid, deliveries, pedestrians and turning traffic dominate and are present all day, not only at the "rush hours".
  The weekday × hour heat-map shows the same slow band between 08:00 and 18:00 on every day, and the weekday/weekend plot shows weekends are faster from 05:00 to 19:00 but not at night.
* **Evening (19:00–23:00):** speed recovers progressively (13.6 → 16.9 km/h) as commuters go home.
* **Caveat:** speed here is *haversine distance ÷ duration*, so it understates the true road speed (real routes are ≈ 1.3× longer than the straight line, see the Manhattan/straight-line ratio of 1.33 in Q2, so true speeds are roughly a quarter higher), but the *pattern* over the day is unaffected.

### Summary of Section 3

| Q | Finding |
|---|---|
| 3.1 | Friday leads on raw totals (220 378) but Saturday leads once the blizzard is removed (≈ 8 645 vs 8 476 per day); the difference is not significant (p = 0.37) – a tie. Monday is least busy. |
| 3.2 | Weekday peak at 18:00–19:00 with a morning commute bump; weekend profile has no morning peak and a busy night (00:00–03:00). Dip/jump at 16:00/17:00 is the taxi shift change. |
| 3.3 | Same total volume, different shape: commuter-driven weekdays vs leisure/night-driven weekends; weekend trips are faster and carry more passengers. |
| 3.4 | Only Memorial Day changes the daily volume markedly (−24 %); other holidays reshape *when* trips happen (weaker commute peaks, earlier/later evening peaks) more than *how many*. |
| 3.5 | Fastest at ≈ 05:00 (≈ 24.6 km/h), slowest through the working day (≈ 12 km/h), steady recovery in the evening. |

---
## 4. Location clusters [12 marks]

Setup for this section: the blizzard days (23–25 Jan) are excluded again so that they do not distort the "typical" spatial patterns, and a hand-made list
of reference landmarks (approximate coordinates, entered by hand – there is no geocoding service offline) is used to *describe* where patterns are.
Heat-maps are 2-D histograms of pickup coordinates (never scatter plots of all 1.4 M points). Clustering uses DBSCAN with the haversine metric as suggested in the brief.

In [ ]:
import folium
from sklearn.cluster import DBSCAN
from sklearn.neighbors import BallTree

nostorm = feat[~feat.pickup_date.isin(STORM)]          # same blizzard exclusion as Section 3
KM_PER_RADIAN = 6371.0088

# Named places used to describe heatmaps and hotspots: (lat, lon). Approximate, hand-entered coordinates.
LANDMARKS = {
    "Times Square": (40.7580, -73.9855), "Penn Station / MSG": (40.7506, -73.9935), "Grand Central": (40.7527, -73.9772),
    "Port Authority": (40.7568, -73.9903), "Columbus Circle": (40.7681, -73.9819), "Lincoln Center": (40.7725, -73.9835),
    "Union Square": (40.7359, -73.9911), "Flatiron / Madison Sq": (40.7411, -73.9897), "Chelsea / Meatpacking": (40.7420, -74.0048),
    "SoHo": (40.7233, -74.0030), "East Village": (40.7265, -73.9815), "Lower East Side": (40.7150, -73.9843),
    "Financial District": (40.7075, -74.0113), "Murray Hill": (40.7479, -73.9757), "Hell's Kitchen": (40.7638, -73.9918),
    "Upper East Side (Lexington)": (40.7736, -73.9566), "Upper West Side": (40.7870, -73.9754), "Harlem (125th St)": (40.8075, -73.9450),
    "Williamsburg": (40.7143, -73.9610), "Long Island City": (40.7447, -73.9485), "Brooklyn Downtown": (40.6928, -73.9903),
    "Greenwich Village / Wash. Sq": (40.7308, -73.9973), "Astor Place / NoHo": (40.7295, -73.9910), "Nolita / Bowery": (40.7210, -73.9935),
    "West Village": (40.7358, -74.0036), "Tribeca": (40.7195, -74.0089), "Chinatown": (40.7158, -73.9970), "Gramercy / Kips Bay": (40.7400, -73.9830),
    "Herald Square": (40.7498, -73.9879), "Fifth Ave / 57th St": (40.7637, -73.9730), "UES Museum Mile (79th St)": (40.7794, -73.9632),
    "Turtle Bay / UN": (40.7517, -73.9690),
    "JFK Airport": (40.6479, -73.7778), "LaGuardia Airport": (40.7769, -73.8740), "Newark Airport": (40.6894, -74.1732),
}
def nearest_landmark(lat, lon, max_km=0.8):
    d = {k: float(haversine_np(lon, lat, v[1], v[0])) for k, v in LANDMARKS.items()}
    k = min(d, key=d.get)
    return (k, d[k]) if d[k] <= max_km else ("(no landmark < %.1f km)" % max_km, d[k])
print(f"{len(nostorm):,} trips used (blizzard days 23-25 Jan excluded)")

### 4.1 Heatmaps [4 marks]

Four groups of pickups are mapped: **weekdays** (Mon–Fri) vs **weekends** (Sat–Sun), and **morning** vs **evening**.
Reasonable hours were taken from Section 3: *morning* = **07:00–10:00** (the commuter peak, 15 % of weekday trips fall here) and *evening* = **18:00–22:00** (the weekday maximum and
the start of the going-out period). Because weekdays have 2.5× more days than weekends, each map shows **mean pickups per day per cell** (≈ 100 m × 120 m cells) on a log colour scale;
the second figure subtracts the *share* of trips in each cell between two groups so differences in **where** (not how much) become visible. The table at the end gives the numbers.

In [ ]:
WD = nostorm[nostorm.day_of_week < 5]; WE = nostorm[nostorm.day_of_week >= 5]
MORN = nostorm[nostorm.pickup_hour.between(7, 9)]            # 07:00-09:59  morning commute
EVE  = nostorm[nostorm.pickup_hour.between(18, 21)]          # 18:00-21:59  evening
groups = {"Weekdays (Mon–Fri)": WD, "Weekends (Sat–Sun)": WE, "Morning 07:00–10:00 (all days)": MORN, "Evening 18:00–22:00 (all days)": EVE}
for k, g in groups.items(): print(f"{k:34s} {len(g):>9,} pickups")

# Manhattan-centred window used for the maps; 1 bin ~ 100 m x 120 m
LON_R, LAT_R = (-74.03, -73.90), (40.68, 40.85)
NB = (130, 190)
def dens(g, rng_=(LON_R, LAT_R), nb=NB):
    H, xe, ye = np.histogram2d(g.pickup_longitude, g.pickup_latitude, bins=nb, range=rng_)
    return H.T, xe, ye
ASPECT = 1 / np.cos(np.radians(40.75))                      # make a degree of longitude look as long as it is on the ground
def show(ax, H, title, cmap="inferno", norm=None, vmin=None, vmax=None):
    im = ax.imshow(H, origin="lower", extent=[*LON_R, *LAT_R], aspect=ASPECT, cmap=cmap, norm=norm, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_title(title, fontsize=10); ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.grid(False)
    for nm in ["Times Square", "Penn Station / MSG", "Grand Central", "Financial District", "Union Square", "Williamsburg"]:
        la, lo = LANDMARKS[nm]; ax.plot(lo, la, "c+", ms=5, alpha=.6)
    return im

# (a) log-density heat-maps, normalised to "mean pickups per day" so groups with a different number of days are comparable
ndays = {"Weekdays (Mon–Fri)": nostorm[nostorm.day_of_week < 5].pickup_date.nunique(), "Weekends (Sat–Sun)": nostorm[nostorm.day_of_week >= 5].pickup_date.nunique(),
         "Morning 07:00–10:00 (all days)": nostorm.pickup_date.nunique(), "Evening 18:00–22:00 (all days)": nostorm.pickup_date.nunique()}
dd = {k: dens(g)[0] / ndays[k] for k, g in groups.items()}
vmax = max(h.max() for h in dd.values())
from matplotlib.colors import LogNorm, TwoSlopeNorm
fig, axes = plt.subplots(2, 2, figsize=(13, 17))
for ax, (k, H) in zip(axes.ravel(), dd.items()):
    im = show(ax, np.where(H > 0, H, np.nan), k, norm=LogNorm(vmin=.02, vmax=vmax))
    plt.colorbar(im, ax=ax, shrink=.6, label="mean pickups per day per cell (log)")
plt.suptitle("Pickup heat-maps (cyan + = reference landmarks)", fontweight="bold", y=.92)
plt.tight_layout(); savefig("q4_1_heatmaps_four"); plt.show()

In [ ]:
# (b) difference maps: share of the group's trips in each cell, group A minus group B (red = relatively more A)
def share(g): H = dens(g)[0]; return H / H.sum()
pairs = [("Weekend − Weekday", WE, WD), ("Evening − Morning", EVE, MORN)]
fig, axes = plt.subplots(1, 2, figsize=(14, 9))
for ax, (nm, A, B) in zip(axes, pairs):
    diff = (share(A) - share(B)) * 1e4
    lim = np.percentile(np.abs(diff), 99.5)
    im = show(ax, diff, f"{nm}: difference in share of pickups", cmap="RdBu_r", norm=TwoSlopeNorm(0, -lim, lim))
    plt.colorbar(im, ax=ax, shrink=.6, label="Δ share (×10⁻⁴ per cell)")
plt.tight_layout(); savefig("q4_1_difference_maps"); plt.show()

In [ ]:
# (c) quantify where each group's pickups are, using circles around named places
REG = {"Midtown (Times Sq, 2 km)": ("Times Square", 2.0), "Penn Station area (0.7 km)": ("Penn Station / MSG", .7), "Grand Central area (0.7 km)": ("Grand Central", .7),
       "Financial District (1.5 km)": ("Financial District", 1.5), "East Village/LES (1.2 km)": ("East Village", 1.2), "Upper East Side (2 km)": ("Upper East Side (Lexington)", 2.0),
       "Williamsburg (1.5 km)": ("Williamsburg", 1.5), "JFK (2 km)": ("JFK Airport", 2.0), "LaGuardia (2 km)": ("LaGuardia Airport", 2.0), "Newark (2 km)": ("Newark Airport", 2.0)}
rows = {}
for gname, g in groups.items():
    r = {}
    for rn, (lm, rad) in REG.items():
        la, lo = LANDMARKS[lm]
        r[rn] = (haversine_np(g.pickup_longitude, g.pickup_latitude, lo, la) < rad).mean() * 100
    r["Manhattan-ish (lon<-73.93 & lat>40.70)"] = ((g.pickup_longitude < -73.93) & (g.pickup_latitude > 40.70) & (g.pickup_longitude > -74.02)).mean() * 100
    rows[gname] = r
share_tbl = pd.DataFrame(rows).round(1)
share_tbl.columns = ["Weekdays", "Weekends", "Morning", "Evening"]
print("% of each group's pickups inside each area"); print(share_tbl.to_string())
print("\nmean distance of pickup from NYC centre (km):", {k: round(g.pickup_dist_centre_km.mean(), 2) for k, g in groups.items()})

**Findings (4.1)** – read together with the four heat-maps, the two difference maps and the table of area shares:

* **The geography is overwhelmingly Manhattan:** ≈ 93 % of pickups in every group fall in Manhattan (south of 110th St, between the rivers); the heat-maps are bright along
  the avenues (long, thin streaks – the avenue grid is clearly visible) and dark in the other boroughs. Only Midtown/Midtown South/Downtown and the two airports (JFK and LaGuardia, ≈ 2 % each) stand out as isolated bright spots;
  Newark is essentially absent (0.0 % – it is in another state and these are NYC cabs) and Brooklyn/Queens/Bronx are sparse – the dataset is a yellow-cab-style sample whose street hails are concentrated in Manhattan
  (outer boroughs are probably served mostly by other vehicle types that are not in this data).
* **Weekdays vs weekends:** the weekday map is concentrated in **Midtown and the East Side**: Midtown (2 km around Times Square) holds 45.0 % of weekday pickups vs 39.9 % at weekends;
  Grand Central 8.9 % vs 6.4 %; the Upper East Side 21.8 % vs 18.1 % – office, business-district and residential-commuter areas. At weekends activity moves **downtown and to nightlife districts**:
  the East Village/Lower East Side share rises from 6.8 % to 10.9 % (+60 %), and in the difference map the strongest red (weekend-heavy) areas are the Lower East Side, East Village, Greenwich Village/West Village and Chelsea/Meatpacking,
  while Midtown East and the Upper East Side are blue (weekday-heavy). LaGuardia's share also drops at weekends (2.8 % → 1.9 %, fewer business flyers).
* **Morning vs evening:** the morning map is dominated by **residential origin areas**: the Upper East Side contributes 26.1 % of morning pickups vs 19.5 % in the evening (people leaving home for work), and Penn Station (8.3 % of morning vs 7.1 % of evening pickups)
  is relatively stronger in the morning. In the evening the pattern is the reverse: pickups concentrate in **Midtown (46.4 % vs 42.1 %) and in the downtown dining/nightlife areas** (East Village/LES 7.2 % vs 6.0 %), i.e.
  commuters leaving offices and people going out. The evening-minus-morning difference map shows red over Midtown West, Chelsea, the Village and the Lower East Side, and blue over the Upper East/West Side.
* **Take-away:** the *when* patterns from Section 3 have a clear *where* counterpart – a **residential → business-district** flow on weekday mornings, **business-district → home/leisure** in the evening, and a **leisure/nightlife**
  geography (downtown, east/west villages) at weekends.

### 4.2 Hotspots [8 marks]

Windows analysed (pooled over all 26 weeks, blizzard dates excluded): **Friday 23:00 → Saturday 02:00** (Friday-night going-out) and **Thursday 17:00 → 20:00** (evening commute).
Hotspots are found with **DBSCAN** on pickup coordinates (haversine metric, ball-tree), i.e. dense groups of pickups with unknown cluster number, exactly as the brief proposes.

**A first, important check – the brief's suggested definition does not work on this data.** The brief suggests a hotspot is "at least 15 pickups in that location in an hour, within 50 or 100 m".
The table below tests this literally: for each individual clock-hour (e.g. Fri 23:00–23:59 on one particular date) we find the largest number of pickups within 50 m and 100 m of any pickup.

In [ ]:
# Two time windows, pooled over all 26 weeks (blizzard excluded).  Friday 23:00 → Saturday 02:00 and Thursday 17:00 → 20:00
dt, dw, hr = nostorm.pickup_datetime, nostorm.pickup_datetime.dt.dayofweek, nostorm.pickup_hour
win_fri = nostorm[((dw == 4) & (hr >= 23)) | ((dw == 5) & (hr < 2))].copy()
win_thu = nostorm[(dw == 3) & (hr >= 17) & (hr < 20)].copy()
windows = {"Fri 23:00 – Sat 02:00": win_fri, "Thu 17:00 – 20:00": win_thu}
for k, g in windows.items():
    g["slot"] = g.pickup_datetime.dt.floor("h")                       # one clock-hour on one date = one "hour" of the brief
    print(f"{k}: {len(g):,} pickups in {g.slot.nunique()} separate clock-hours -> {len(g)/g.slot.nunique():.0f} pickups/hour city-wide")

# Can the brief's literal definition (>= 15 pickups within 50/100 m in ONE hour) ever be met in this data?
lit = []
for k, g in windows.items():
    for r in (50, 100):
        best = 0; nhot = 0
        for _, gg in g.groupby("slot"):
            X = np.radians(gg[["pickup_latitude", "pickup_longitude"]].values)
            c = BallTree(X, metric="haversine").query_radius(X, r / 1000 / KM_PER_RADIAN, count_only=True)
            best = max(best, c.max()); nhot += int((c >= 15).any())
        lit.append({"window": k, "radius_m": r, "max pickups within radius in a single hour": int(best), "hours with a >=15 hotspot": nhot, "hours": g.slot.nunique()})
pd.DataFrame(lit)

**Result of the check:** in *no single hour* of either window do 15 pickups fall within 100 m of each other (the maximum is only 14 within 100 m and 10–11 within 50 m); hence, taken literally, the
definition finds **zero hotspots**. The reason is that this training file is a *sample* of taxi activity (only ≈ 444–485 pickups per hour for the whole of New York in these windows, far fewer than a real hour of NYC taxi demand), so absolute thresholds from the brief are too high.
We therefore keep the *idea* of the definition (a hotspot is a place with a persistent rate of pickups per hour) but express it as a **rate** and apply it to the pooled data:

> `min_samples = (required pickups per hour) × (number of clock-hours pooled)` with `eps` = 50 m.

**Motivation of the parameters**
* **ε = 50 m.** NYC blocks are ≈ 80 m (north–south) × 270 m (east–west), and consumer GPS is accurate to ≈ 10–50 m, so 50 m is about one corner or half a short block: it is the smallest neighbourhood that is robust to GPS noise,
  and it gives a hotspot a location a driver can actually drive to. 100 m was also tried: the sensitivity table shows that with 100 m (and a low rate) DBSCAN **chains neighbouring blocks together** – the largest "cluster" has a radius of 1.2–1.5 km, i.e. most of Midtown becomes one blob, which is useless for deciding where to wait.
* **Rate = 1 pickup per hour on average** (min_samples = 76 for Friday night, 78 for Thursday). At 0.5/hour clusters are numerous (>100) and begin to chain (radius up to 600 m); at 2/hour almost nothing survives with 50 m (6 clusters). 1/hour is the point at which clusters are compact
  (largest radius < 200 m), still numerous enough to be informative, and well above the typical background (the *median* pickup has only ≈ 0.3 other pickups per hour within 50 m). In practice it means "a cab waiting here would on average see a new fare at least once an hour" – a low bar in
  absolute terms, but a high one relative to this sample (which is only a fraction of real activity).
* The **sensitivity grid** below (ε ∈ {30, 50, 75, 100} m × rate ∈ {0.5, 1, 2}) shows how the cluster count and the largest radius depend on both parameters.

In [ ]:
def run_dbscan(g, eps_m, per_hour):
    """DBSCAN on pickup coordinates (haversine). min_samples = per_hour x number of clock-hours pooled."""
    n_slots = g.slot.nunique(); ms = int(round(per_hour * n_slots))
    X = np.radians(g[["pickup_latitude", "pickup_longitude"]].values)
    db = DBSCAN(eps=eps_m / 1000 / KM_PER_RADIAN, min_samples=ms, algorithm="ball_tree", metric="haversine").fit(X)
    return db.labels_, ms

# sensitivity grid: how many clusters, how much noise, and how spread-out is the largest cluster (chaining check)
grid = []
for k, g in windows.items():
    X = g[["pickup_latitude", "pickup_longitude"]].values
    for eps_m in (30, 50, 75, 100):
        for ph in (0.5, 1, 2):
            lab, ms = run_dbscan(g, eps_m, ph)
            ext = [haversine_np(X[lab == c, 1], X[lab == c, 0], X[lab == c, 1].mean(), X[lab == c, 0].mean()).max() * 1000 for c in range(lab.max() + 1)]
            grid.append({"window": k, "eps_m": eps_m, "pickups_per_hour": ph, "min_samples": ms, "clusters": int(lab.max() + 1), "noise_%": round((lab == -1).mean() * 100, 1),
                         "largest_cluster_radius_m": round(max(ext)) if ext else 0})
grid = pd.DataFrame(grid)
print("Number of clusters"); print(grid.pivot_table(index=["window", "eps_m"], columns="pickups_per_hour", values="clusters").to_string())
print("\nRadius (m) of the largest cluster – large values mean DBSCAN is chaining neighbouring blocks together")
grid.pivot_table(index=["window", "eps_m"], columns="pickups_per_hour", values="largest_cluster_radius_m")

In [ ]:
EPS_M, PER_HOUR = 50, 1                    # chosen parameters (justified above)
res = {}
for k, g in windows.items():
    lab, ms = run_dbscan(g, EPS_M, PER_HOUR)
    g["cluster"] = lab
    n_slots = g.slot.nunique()
    rows = []
    for c in range(lab.max() + 1):
        pts = g[g.cluster == c]
        la, lo = pts.pickup_latitude.mean(), pts.pickup_longitude.mean()
        # extent: largest distance of a member from the centroid
        ext = float(haversine_np(pts.pickup_longitude.values, pts.pickup_latitude.values, lo, la).max()) * 1000
        nm, dist = nearest_landmark(la, lo)
        rows.append({"cluster": c, "pickups": len(pts), "pickups_per_hour": len(pts) / n_slots, "lat": round(la, 4), "lon": round(lo, 4), "max_radius_m": ext, "nearest_landmark": nm, "landmark_dist_m": dist * 1000})
    tbl = pd.DataFrame(rows).sort_values("pickups", ascending=False).reset_index(drop=True)
    res[k] = (g, tbl, ms)
    print(f"{k}: eps = {EPS_M} m, min_samples = {ms} -> {len(tbl)} clusters; {(lab == -1).mean():.1%} of pickups are noise; "
          f"clustered pickups = {(lab >= 0).sum():,} ({(lab >= 0).mean():.1%})")
res["Fri 23:00 – Sat 02:00"][1].head(12).round(2)

In [ ]:
res["Thu 17:00 – 20:00"][1].head(12).round(2)

In [ ]:
# static maps (used in the PDF) – noise in grey, clusters coloured, top clusters numbered
fig, axes = plt.subplots(1, 2, figsize=(17, 11))
for ax, (k, (g, tbl, ms)) in zip(axes, res.items()):
    noise = g[g.cluster < 0]; cl = g[g.cluster >= 0]
    ax.scatter(noise.pickup_longitude, noise.pickup_latitude, s=1, c="#bbbbbb", alpha=.5, rasterized=True)
    cmap = plt.get_cmap("tab20")
    ax.scatter(cl.pickup_longitude, cl.pickup_latitude, s=3, c=[cmap(c % 20) for c in cl.cluster], rasterized=True)
    for i, r in tbl.head(10).iterrows():
        ax.annotate(f"{i+1}", (r.lon, r.lat), fontsize=9, fontweight="bold", color="k", xytext=(4, 4), textcoords="offset points",
                    bbox=dict(boxstyle="circle,pad=.15", fc="yellow", ec="k", lw=.5))
    ax.set_xlim(-74.03, -73.93); ax.set_ylim(40.69, 40.82); ax.set_aspect(ASPECT)
    ax.set_title(f"{k}\n{len(tbl)} hotspot clusters (eps={EPS_M} m, min_samples={ms})"); ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
plt.tight_layout(); savefig("q4_2_hotspot_maps"); plt.show()

In [ ]:
# interactive folium maps (also saved as HTML in maps/); circle area ~ pickups per hour
os.makedirs("maps", exist_ok=True)
def folium_map(k, g, tbl, fname):
    m = folium.Map(location=[40.7450, -73.9850], zoom_start=12, tiles="cartodbpositron")
    folium.Circle(location=[40.7580, -73.9855], radius=1, color="red", fill=True, popup="Times Square").add_to(m)
    for i, r in tbl.iterrows():
        folium.Circle(location=[r.lat, r.lon], radius=max(r.max_radius_m, 60), color="crimson" if i < 10 else "royalblue", fill=True, fill_opacity=.35, weight=1,
                      popup=folium.Popup(f"#{i+1} {r.nearest_landmark}<br>{r.pickups:.0f} pickups ({r.pickups_per_hour:.1f}/hour)", max_width=250)).add_to(m)
        if i < 10:
            folium.Marker([r.lat, r.lon], icon=folium.DivIcon(html=f'<div style="font-size:11pt;font-weight:bold;color:black;background:yellow;border-radius:50%;padding:0 4px">{i+1}</div>')).add_to(m)
    m.save(f"maps/{fname}.html")
    return m
m_fri = folium_map(*("Fri 23:00 – Sat 02:00", *res["Fri 23:00 – Sat 02:00"][:2]), "hotspots_fri_night")
m_thu = folium_map(*("Thu 17:00 – 20:00", *res["Thu 17:00 – 20:00"][:2]), "hotspots_thu_evening")
m_fri

In [ ]:
m_thu

In [ ]:
# compare the two windows: which hotspot locations are shared?
a, b = res["Fri 23:00 – Sat 02:00"][1], res["Thu 17:00 – 20:00"][1]
D_ = haversine_np(a.lon.values[:, None], a.lat.values[:, None], b.lon.values[None, :], b.lat.values[None, :])
shared_a = (D_.min(axis=1) < .25)
print(f"Fri-night clusters that have a Thursday-evening cluster within 250 m: {shared_a.sum()} of {len(a)}")
print(f"Thursday-evening clusters that have a Fri-night cluster within 250 m: {(D_.min(axis=0) < .25).sum()} of {len(b)}")
cmp_ = pd.concat({"Fri night top 10": a.head(10)[["nearest_landmark", "pickups_per_hour"]].reset_index(drop=True),
                  "Thu evening top 10": b.head(10)[["nearest_landmark", "pickups_per_hour"]].reset_index(drop=True)}, axis=1).round(2)
cmp_

In [ ]:
# where do the two windows differ?  share of each window's clustered pickups per area
def area_share(g):
    c = g[g.cluster >= 0]
    out = {}
    for rn, (lm, rad) in {"Midtown W (Times Sq/Port Auth/Penn, 1.5 km)": ("Times Square", 1.5), "Grand Central/Murray Hill (1 km)": ("Grand Central", 1.0),
                          "Downtown nightlife (E. Village/LES/SoHo/Meatpacking)": ("East Village", 1.8), "Financial District (1 km)": ("Financial District", 1.0),
                          "Williamsburg (1.5 km)": ("Williamsburg", 1.5), "Upper East/West Side": ("Upper East Side (Lexington)", 2.0)}.items():
        la, lo = LANDMARKS[lm]; out[rn] = (haversine_np(c.pickup_longitude, c.pickup_latitude, lo, la) < rad).mean() * 100
    return out
pd.DataFrame({k: area_share(v[0]) for k, v in res.items()}).round(1)

**How many clusters did we find?** With ε = 50 m and ≥ 1 pickup/hour (min_samples 76/78):
**36 hotspot clusters for Friday 23:00–Saturday 02:00** and **22 clusters for Thursday 17:00–20:00**.
They are compact (radius under 200 m, i.e. one or two street corners) and together contain 18.5 % and 10.4 % of the window's pickups respectively; the remaining pickups are classified as noise, which means demand is mostly diffuse
and only a minority of fares come from persistent hotspot corners. (With 100 m and 2 pickups/hour the counts would be 21 and 18, see the grid; the numbers depend on the parameters, so the cluster count should be read as "a few dozen" rather than a precise figure.)

**Where are the hotspots?** (landmark names are the nearest of the hand-entered reference points, so are approximate; see the maps and the tables of the top clusters)
* **Friday night (23:00–02:00): downtown nightlife.** 52.6 % of the clustered pickups lie in the Village/East Village/Lower East Side/Meatpacking region against 3.9 % for Thursday evening. The busiest clusters are
  around Astor Place/NoHo, the Lower East Side/Nolita–Bowery area (several clusters), the Meatpacking District/West Chelsea (three clusters), the West Village, and the East Village (≈ 2.7–5.2 pickups per hour each),
  plus Penn Station/Madison Square Garden (3.5/hour). A JFK cluster (2.4/hour) is also present: late-night airport arrivals.
* **Thursday evening (17:00–20:00): transport hubs and airports.** Midtown West (Penn Station/MSG, Port Authority, Columbus Circle, Fifth Avenue/57th Street) carries 40.7 % of the clustered pickups (Friday night: 15.9 %),
  the Upper East/West Side 21.1 % (vs 0 % on Friday night) – commuters. The single biggest hotspot is the **LaGuardia** terminal area (6.9/hour, plus two more LaGuardia clusters), followed by two Penn Station clusters (5.3 and 3.9/hour) and the JFK terminals.
* **The two sets of hotspots are largely different:** only 7 of 36 Friday-night clusters have a Thursday-evening cluster within 250 m (8 of 22 the other way round); the shared ones are mostly transport hubs (Penn Station, airports).

**Advice for a driver:** on Thursday early evening, wait at (or near) the airports' terminals and the Penn Station/Port Authority/Columbus Circle hubs; on Friday night after 23:00 position the cab in the downtown nightlife triangle
(East Village–Lower East Side–Meatpacking–West Village). In both windows the hotspots are small, so the driver should know the exact corners listed in the cluster tables.

**Limitations:** the pooled analysis treats all 25–26 occurrences alike (weather or events are averaged out); DBSCAN with a single ε cannot describe both dense Midtown and sparse outer areas (hence 80–90 % of pickups are labelled noise);
the landmark labels are approximate; and pickups include flows that are *not* hotspots in the driver's sense (e.g. airports are hotspots only for one direction of travel).

### Summary of Section 4

| Q | Finding |
|---|---|
| 4.1 | ≈ 93 % of pickups are in Manhattan. Weekdays are Midtown/East-Side/Grand-Central heavy; weekends shift downtown to nightlife areas (East Village/LES share 6.8 % → 10.9 %). Mornings come from residential areas (Upper East Side 26 %), evenings from Midtown and downtown. |
| 4.2 | The literal "≥ 15 pickups per hour within 50–100 m" definition is never met in this sampled data (max 14 within 100 m). Using a rate (≥ 1 pickup/hour pooled) with DBSCAN ε = 50 m we find **36 hotspots on Friday 23:00–02:00** (downtown nightlife) and **22 on Thursday 17:00–20:00** (airports and Midtown transport hubs). |

---
## 5. Airports [9 marks]

**Task:** the average travel time from the Empire State Building (ESB) to JFK, by time of day, and how it compares with Newark.

**Method.** A trip is an "ESB → airport" trip if its **pickup** lies within a radius `R_ESB` of the ESB *and* its **dropoff** lies within a radius `R_air` of the airport centre
(great-circle distance, `haversine_np`). Travel time is the recorded trip duration of the *cleaned* data (Section 1), and the time of day is the pickup hour.

*Note on the coordinates:* the brief labels the three points "(long, lat)", but the values `(40.647929, −73.777813)` etc. are clearly **(latitude, longitude)** – 40.6° N, 73.8° W is JFK,
whereas the reverse order would put JFK in the Arctic/Indian Ocean – so they are used as (lat, lon). The straight-line distances printed below confirm the geometry:
ESB–JFK ≈ 21 km, ESB–Newark ≈ 18 km.

In [ ]:
from scipy import stats

# Coordinates from the brief. The brief labels them "(long, lat)" but the numbers are clearly (lat, lon): 40.65 N / -73.78 E is JFK.
PLACES = {"JFK": (40.647929, -73.777813), "Empire State Building": (40.756724, -73.983806), "Newark": (40.689442, -74.173242)}   # (lat, lon)
def dist_to(df_, place, which):           # which = "pickup" or "dropoff"; returns km
    la, lo = PLACES[place]
    return haversine_np(df_[f"{which}_longitude"].values, df_[f"{which}_latitude"].values, lo, la)

d_esb_pick = dist_to(feat, "Empire State Building", "pickup")
d_jfk_drop, d_ewr_drop = dist_to(feat, "JFK", "dropoff"), dist_to(feat, "Newark", "dropoff")
print("Straight-line distances between the three places (km): ESB-JFK %.1f, ESB-Newark %.1f, JFK-Newark %.1f" % (
    haversine_np(PLACES["JFK"][1], PLACES["JFK"][0], PLACES["Empire State Building"][1], PLACES["Empire State Building"][0]),
    haversine_np(PLACES["Newark"][1], PLACES["Newark"][0], PLACES["Empire State Building"][1], PLACES["Empire State Building"][0]),
    haversine_np(PLACES["Newark"][1], PLACES["Newark"][0], PLACES["JFK"][1], PLACES["JFK"][0])))

**Choosing the radii.** The radius must be large enough to contain the whole area where a taxi can legitimately pick up / drop off for that place, but small enough not to include other places.
* **Airports (R_air = 2 km).** An airport is not a point: the terminals, access roads and car parks span 1–2 km. The *left* plot shows that the number of drop-offs rises steeply with radius up to ≈ 1.2 km for JFK
  (and ≈ 1 km for Newark) and then almost flattens; the *middle* plot (drop-offs per km² in rings of increasing radius) shows the density collapsing by one to two orders of magnitude at the same radius.
  A radius of **2 km** therefore contains the whole terminal area with a safety margin for GPS error (the taxi might stop on a service road) and adds only ≈ 600 more JFK drop-offs than 1.2 km. For Newark it is equally conservative.
  A smaller radius (1 km) would cut off terminals; a larger one would begin to include unrelated places (e.g. neighbourhoods around JFK) – so ≈ 2 km is a sensible compromise.
* **Empire State Building (R_ESB = 0.5 km).** The ESB is a single building, but passengers hail cabs in the surrounding blocks (Herald Square, Penn Station, Fifth Avenue) and the recorded GPS position is the position of the cab when the meter starts.
  500 m (about two long or six short blocks) captures a sensible neighbourhood while keeping all pickups in the Midtown core where traffic conditions are essentially the same; it gives enough trips to estimate hourly means (862 JFK and 236 Newark trips).
* **Robustness.** The sensitivity table (and the right-hand plot) shows that the *median* travel time is hardly affected by the choice: for JFK it stays between **45.1 and 47.6 minutes** for every combination of radii (ESB 0.2–1.0 km, airport 1–3 km),
  for Newark between **32.7 and 35.8 minutes**. Only the sample size changes (JFK from 116 to 3 123 trips). The conclusions below therefore do not depend on the exact radius.

In [ ]:
radii = np.arange(0.1, 6.01, 0.1)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
for nm, d, c in [("JFK", d_jfk_drop, "C0"), ("Newark", d_ewr_drop, "C3")]:
    cum = np.array([(d < r).sum() for r in radii])
    axes[0].plot(radii, cum, color=c, label=nm)
    axes[1].plot(radii[1:], np.diff(cum) / np.diff(radii**2 * np.pi), color=c, label=nm)       # drop-offs per km² in each ring
axes[0].axvline(2, color="k", ls="--", lw=1); axes[0].set_xlabel("radius around the airport centre (km)"); axes[0].set_ylabel("# trips whose drop-off lies inside")
axes[0].set_title("Cumulative drop-offs vs radius"); axes[0].legend()
axes[1].set_yscale("log"); axes[1].axvline(2, color="k", ls="--", lw=1); axes[1].set_xlabel("radius (km)"); axes[1].set_ylabel("drop-offs per km² in the ring (log)")
axes[1].set_title("Drop-off density by ring: drops sharply beyond ≈ 1–1.2 km"); axes[1].legend()

rad_esb = np.arange(0.15, 1.51, 0.05)
for nm, d, c in [("JFK", d_jfk_drop, "C0"), ("Newark", d_ewr_drop, "C3")]:
    med = [feat.loc[(d_esb_pick < r) & (d < 2.0), "duration_min"].median() for r in rad_esb]
    nn = np.array([((d_esb_pick < r) & (d < 2.0)).sum() for r in rad_esb])
    axes[2].plot(rad_esb, med, color=c, label=f"{nm} (median, min)")
    axes[2].fill_between(rad_esb, med, alpha=0)
axes[2].axvline(.5, color="k", ls="--", lw=1); axes[2].set_xlabel("radius around the Empire State Building (km)"); axes[2].set_ylabel("median travel time (min)")
axes[2].set_title("Median travel time is stable in the radius"); axes[2].legend()
plt.tight_layout(); savefig("q5_radius_choice"); plt.show()

# how much do the answers depend on the radius?  (median travel time of ESB -> airport trips)
rows = []
for re_ in (0.2, 0.3, 0.5, 0.75, 1.0):
    for ra in (1.0, 1.5, 2.0, 3.0):
        for nm, d in (("JFK", d_jfk_drop), ("Newark", d_ewr_drop)):
            sel = (d_esb_pick < re_) & (d < ra)
            rows.append({"ESB_radius_km": re_, "airport_radius_km": ra, "airport": nm, "n": int(sel.sum()), "median_min": round(feat.loc[sel, "duration_min"].median(), 1)})
sens = pd.DataFrame(rows).pivot_table(index=["ESB_radius_km", "airport_radius_km"], columns="airport", values=["n", "median_min"])
sens

**Results.** The tables below give the overall statistics for the chosen radii, then the hour-by-hour travel time and the comparison between the two airports.

In [ ]:
R_ESB, R_AIR = 0.5, 2.0                                     # chosen radii in km
esb_jfk = feat[(d_esb_pick < R_ESB) & (d_jfk_drop < R_AIR)].copy()
esb_ewr = feat[(d_esb_pick < R_ESB) & (d_ewr_drop < R_AIR)].copy()
trips = {"JFK": esb_jfk, "Newark": esb_ewr}

def summ(g):
    return pd.Series({"trips": len(g), "mean (min)": g.duration_min.mean(), "median (min)": g.duration_min.median(), "std (min)": g.duration_min.std(),
                      "25th pct": g.duration_min.quantile(.25), "75th pct": g.duration_min.quantile(.75), "mean dist (km, straight line)": g.distance_km.mean(),
                      "mean speed (km/h)": g.speed_kmh.mean()})
overall = pd.DataFrame({k: summ(g) for k, g in trips.items()}).T.round(1)
print(overall.to_string())
u, p = stats.mannwhitneyu(esb_jfk.duration_min, esb_ewr.duration_min)
print(f"\nMann–Whitney U test JFK vs Newark durations: p = {p:.2e}")
print("Date coverage of these trips:", esb_jfk.pickup_date.min().date(), "→", esb_jfk.pickup_date.max().date())
for k, g in trips.items():
    print(f"{k}: weekday median {g[~g.is_weekend].duration_min.median():.1f} min (n={int((~g.is_weekend).sum())}),  weekend median {g[g.is_weekend].duration_min.median():.1f} min (n={int(g.is_weekend.sum())})")
overall

In [ ]:
# travel time by time of day (hour of pickup)
def hourly_stats(g):
    gr = g.groupby("pickup_hour").duration_min
    out = pd.DataFrame({"n": gr.size(), "mean": gr.mean(), "median": gr.median(), "q25": gr.quantile(.25), "q75": gr.quantile(.75)}).reindex(range(24))
    # bootstrap 95 % CI of the mean (only where n >= 5)
    lo, hi = [], []
    for h in range(24):
        x = g.loc[g.pickup_hour == h, "duration_min"].values
        if len(x) >= 5:
            bs = rng.choice(x, size=(1000, len(x))).mean(axis=1); lo.append(np.percentile(bs, 2.5)); hi.append(np.percentile(bs, 97.5))
        else: lo.append(np.nan); hi.append(np.nan)
    out["ci_lo"], out["ci_hi"] = lo, hi
    return out
hs = {k: hourly_stats(g) for k, g in trips.items()}

fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for j, (k, c) in enumerate([("JFK", "C0"), ("Newark", "C3")]):
    h = hs[k]; ax = axes[0, j]
    ax.fill_between(h.index, h.q25, h.q75, color=c, alpha=.15, label="interquartile range")
    ax.fill_between(h.index, h.ci_lo, h.ci_hi, color=c, alpha=.35, label="95 % bootstrap CI of mean")
    ax.plot(h.index, h["mean"], "-o", color=c, label="mean"); ax.plot(h.index, h["median"], "--", color="k", lw=1.2, label="median")
    ax.set_ylabel("travel time (minutes)"); ax.set_title(f"Empire State Building → {k} Airport"); ax.legend(fontsize=8, loc="upper left"); ax.set_ylim(10, 100)
    axes[1, j].bar(h.index, h.n, color=c, alpha=.7); axes[1, j].set_ylabel("# trips"); axes[1, j].set_xlabel("pickup hour"); axes[1, j].set_xticks(range(0, 24, 2))
plt.tight_layout(); savefig("q5_travel_time_by_hour"); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.8))
for k, c in [("JFK", "C0"), ("Newark", "C3")]:
    ax.plot(hs[k].index, hs[k]["mean"], "-o", color=c, label=f"{k} (mean)")
    ax.fill_between(hs[k].index, hs[k].ci_lo, hs[k].ci_hi, color=c, alpha=.2)
ax.set_xlabel("pickup hour"); ax.set_ylabel("mean travel time (min)"); ax.set_xticks(range(0, 24, 2)); ax.legend(); ax.set_title("JFK vs Newark: travel time from the Empire State Building")
plt.tight_layout(); savefig("q5_jfk_vs_newark"); plt.show()
pd.concat({k: hs[k][["n", "mean", "median"]].round(1) for k in hs}, axis=1).T

In [ ]:
# coarser periods give larger samples than single hours
PERIODS = [("Night 00-05", 0, 5), ("Morning rush 06-09", 6, 9), ("Midday 10-15", 10, 15), ("Afternoon rush 16-19", 16, 19), ("Evening 20-23", 20, 23)]
rows = []
for k, g in trips.items():
    for nm, a, b in PERIODS:
        x = g[g.pickup_hour.between(a, b)].duration_min
        rows.append({"airport": k, "period": nm, "n": len(x), "mean": x.mean(), "median": x.median(), "q25": x.quantile(.25), "q75": x.quantile(.75)})
per = pd.DataFrame(rows).round(1)
per_pivot = per.pivot(index="period", columns="airport", values=["n", "median", "mean"]).loc[[p[0] for p in PERIODS]]
print(per_pivot.to_string())
fig, ax = plt.subplots(figsize=(9, 4.5))
w = .38; xs = np.arange(len(PERIODS))
for i, (k, c) in enumerate([("JFK", "C0"), ("Newark", "C3")]):
    s_ = per[per.airport == k].set_index("period").loc[[p[0] for p in PERIODS]]
    ax.bar(xs + (i - .5) * w, s_["median"], w, color=c, label=k, yerr=[s_["median"] - s_["q25"], s_["q75"] - s_["median"]], capsize=3, alpha=.8)
ax.set_xticks(xs); ax.set_xticklabels([p[0] for p in PERIODS], rotation=15); ax.set_ylabel("median travel time (min); whiskers = IQR"); ax.legend(); ax.set_title("Travel time by period of the day")
plt.tight_layout(); savefig("q5_periods"); plt.show()
per_pivot

**Findings (Q5)**

*Overall averages (R_ESB = 0.5 km, R_air = 2 km).*
* **ESB → JFK:** **mean 49.1 min, median 46.0 min** (interquartile range 36.6–59.9 min, std 17.1 min) from 862 trips; mean straight-line distance 20.9 km, mean speed 28.6 km/h.
* **ESB → Newark:** **mean 36.3 min, median 33.1 min** (IQR 28.3–40.9 min) from 236 trips; distance 17.9 km, speed 31.8 km/h.
* **Comparison:** JFK takes ≈ **13 minutes (≈ 35–40 %) longer** than Newark on average (46.0 vs 33.1 min median; Mann–Whitney p ≈ 10⁻³¹, so the difference is highly significant). Part of it is distance (ESB is 21 km from JFK but 18 km from Newark in a straight line, ≈ 20 % further),
  and part is speed (JFK trips average 28.6 km/h vs 31.8 km/h for Newark; plausibly the JFK route crosses Queens' congested expressways, while the Newark route goes through a tunnel and the New Jersey highways – the data do not tell us the routes, so this is only a likely explanation). JFK is also more variable (std 17 vs 12 min).

*Travel time by time of day* (hourly plots and period table):
* **JFK:** the fastest trips are **night and early morning (01:00–05:00): ≈ 28–33 min** (median in the 00–05 period 28.6 min). Travel time rises slowly during the morning (≈ 33 min at 06:00, 38–41 min at 07:00–10:00) and midday
  (≈ 43–46 min at 11:00–13:00), then **jumps at 14:00 (≈ 58 min) and peaks at 16:00 – ≈ 66 min mean** (more than double the night time) – the afternoon/evening rush hour – and falls back to ≈ 43 min by 19:00 and ≈ 37–47 min late evening
  (the late-evening hours have few trips: 6–22 per hour from 19:00 to 23:00, so those points are noisy; look at the confidence bands). Period medians: night 28.6, morning rush (06–09) 35.4, midday (10–15) 48.7, **afternoon rush (16–19) 59.6**, evening (20–23) 39.8 min.
* **Newark:** the same shape at a lower level: ≈ 26–29 min at 04:00–06:00, 32–37 min in the morning and midday, 42–45 min at 14:00–16:00 and **a peak at 17:00 (≈ 59 min)**, followed by a decline. The 17:00 point is based on only **4 trips**, so it is suggestive but not reliable;
  period medians: night 27.1, morning 32.8, midday 34.0, afternoon rush 42.0, evening 32.7 min. Newark has few trips before 04:00 and after 20:00 (no estimate where n < 5, shown as gaps), so its night/evening estimates are the weakest.
* **Relative effect of traffic:** the afternoon rush hour costs ≈ 2.1× the night travel time to JFK (59.6 vs 28.6 min) but ≈ 1.55× to Newark (42.0 vs 27.1 min) – the JFK route is more affected by the rush hour, even though the Newark route also has an afternoon peak.
* **Weekday vs weekend:** median 50.4 vs 39.9 min for JFK (n = 605 / 257) and 34.0 vs 30.6 min for Newark (n = 182 / 54): weekends are faster, in line with the lower congestion seen in Section 3.5. Because the hourly curves mix weekdays and weekends, part of the afternoon
  rise (more weekday trips in the afternoon) is due to the weekday congestion.

**Practical conclusion:** to reach JFK from the Empire State Building allow **≈ 45 min on average, but ≈ 1 hour in the afternoon rush (14:00–18:00) and only ≈ 30 min late at night or early in the morning**; for Newark allow **≈ 33 min on average, ≈ 45 min at 14:00–17:00 and ≈ 27 min early in the morning**.

**Caveats:** the number of trips per hour is modest (and small for Newark at night and in the evening), so single hours should be interpreted through the confidence intervals; trips in the data are only those that *chose a taxi* for this route; travel times are recorded
meter times (door-to-door within the taxi) and do not include waiting time; and 2016 traffic conditions are baked in.

---
## 6. Dimensionality reduction [22 marks]

Each trip is described by many features. PCA (linear) and t-SNE / UMAP (non-linear) are used to look for natural "types" of trip.
As required, everything is done on a **random sample of 20 000 trips from the cleaned data, with a fixed seed (42)** so that results are reproducible.
(Embeddings are cached in `cache/` so re-running the notebook is fast; delete the folder to recompute from scratch. Runtimes quoted below are from the original run on 4 CPU cores.)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
import umap, pickle, time

CACHE_DIR = "cache"; os.makedirs(CACHE_DIR, exist_ok=True)     # embeddings are cached so the notebook can be re-run quickly (delete cache/ to recompute)
def cached(name, fn):
    path = os.path.join(CACHE_DIR, name + ".pkl")
    if os.path.exists(path):
        with open(path, "rb") as fh: return pickle.load(fh)
    out = fn()
    with open(path, "wb") as fh: pickle.dump(out, fh)
    return out

N_SAMPLE = 20_000
sample = feat.sample(n=N_SAMPLE, random_state=SEED).reset_index(drop=True)      # fixed seed -> reproducible sample of the CLEANED data
print(f"Sample: {len(sample):,} of {len(feat):,} cleaned trips ({len(sample)/len(feat):.2%}), seed = {SEED}")

### 6.1 Feature matrix [2 marks]

| Feature(s) | Transformation | Justification |
|---|---|---|
| `pickup_lon`, `pickup_lat`, `dropoff_lon`, `dropoff_lat` | none (standardised) | *where* the trip starts and ends; the brief's required spatial features. Lon/lat are used directly because the study area is small (≈ 30 km), so the Earth's curvature is negligible. |
| `log_distance_km` | `log` | trip length; raw distance is strongly right-skewed (skewness 2.8 → 0.37 after the log), so a few long airport trips would otherwise dominate every Euclidean distance. |
| `log_duration_s` | `log` | duration, same reason (skewness 2.15 → −0.13). |
| `log_speed_kmh` | `log` | average speed (a derived feature; skewness 1.46 → −0.20). |
| `hour_sin`, `hour_cos` | `sin(2πh/24)`, `cos(2πh/24)` | **cyclic encoding of the pickup hour**: 23:00 and 00:00 are neighbours, but are 23 apart if the hour is used as an integer. |
| `dow_sin`, `dow_cos` | `sin(2πd/7)`, `cos(2πd/7)` | day of week, encoded cyclically for the same reason (Sunday is next to Monday). |

All features are then **standardised** (`StandardScaler`: mean 0, standard deviation 1). Scaling is essential because PCA, t-SNE and UMAP (and DBSCAN) are all based on Euclidean distances / variances:
without it, features with large numeric ranges (duration in seconds, up to 10 000) would swamp coordinates in degrees (range ≈ 0.1) and the unit-circle hour encodings. Standardising the sin/cos pairs is acceptable – the two coordinates then carry equal weight –
and a standard scaler is preferred to min–max scaling because the features contain some remaining outliers that min–max would compress.
The final matrix is 20 000 × 11. The correlation plot shows that the features are not redundant apart from the expected relations (distance–duration–speed, pickup–dropoff coordinates, sin/cos pairs).

**A consequence worth knowing in advance:** the day of week takes only seven distinct values, so `dow_sin`/`dow_cos` place every trip on one of seven fixed points; this will leave a strong imprint on all embeddings (see 6.3–6.4).

In [ ]:
# ---- 6.1 feature matrix -------------------------------------------------------------------------------------------
S = sample
FE = pd.DataFrame({
    "pickup_lon": S.pickup_longitude, "pickup_lat": S.pickup_latitude,
    "dropoff_lon": S.dropoff_longitude, "dropoff_lat": S.dropoff_latitude,
    "log_distance_km": np.log(S.distance_km), "log_duration_s": np.log(S.trip_duration), "log_speed_kmh": np.log(S.speed_kmh),
    "hour_sin": np.sin(2 * np.pi * S.pickup_hour / 24), "hour_cos": np.cos(2 * np.pi * S.pickup_hour / 24),
    "dow_sin": np.sin(2 * np.pi * S.day_of_week / 7), "dow_cos": np.cos(2 * np.pi * S.day_of_week / 7),
})
FEATURES = list(FE.columns)
scaler = StandardScaler()
X = scaler.fit_transform(FE.values)
print("Feature matrix:", X.shape, "| mean ≈ 0:", np.allclose(X.mean(0), 0, atol=1e-8), "| std = 1:", np.allclose(X.std(0), 1))

# skewness before / after the log transform shows why the log is used
raw_cols = pd.DataFrame({"distance_km": S.distance_km, "duration_s": S.trip_duration, "speed_kmh": S.speed_kmh})
skew = pd.DataFrame({"skew raw": raw_cols.skew().values, "skew after log": FE[["log_distance_km", "log_duration_s", "log_speed_kmh"]].skew().values}, index=raw_cols.columns).round(2)
print(skew.to_string())

# colour variables used in every embedding plot
hour = S.pickup_hour.values; weekend = S.is_weekend.values.astype(int); dist = S.distance_km.values
dist_col = np.clip(dist, None, np.percentile(dist, 99))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.violinplot(data=FE, ax=axes[0], orient="h", inner=None, linewidth=.6, color="#9ecae1", density_norm="width"); axes[0].set_title("Features before scaling (different units/ranges)")
sns.violinplot(data=pd.DataFrame(X, columns=FEATURES), ax=axes[1], orient="h", inner=None, linewidth=.6, color="#a1d99b", density_norm="width"); axes[1].set_title("After StandardScaler (mean 0, sd 1)")
plt.tight_layout(); savefig("q6_1_features"); plt.show()
corr = pd.DataFrame(X, columns=FEATURES).corr()
fig, ax = plt.subplots(figsize=(8, 6.5)); sns.heatmap(corr, cmap="RdBu_r", center=0, annot=True, fmt=".2f", annot_kws={"size": 7}, ax=ax); ax.set_title("Correlation of the scaled features")
plt.tight_layout(); savefig("q6_1_feature_correlation"); plt.show()
skew

### 6.2 PCA [4 marks]

PCA is fitted on all 11 standardised features. The scree plot shows each component's explained variance ratio and the cumulative curve; the first two components are then plotted coloured by (i) hour, (ii) weekday/weekend and (iii) trip distance,
and the loadings of the leading components are inspected.

In [ ]:
# ---- 6.2 PCA ------------------------------------------------------------------------------------------------------
t0 = time.time(); pca = PCA().fit(X); Z = pca.transform(X); t_pca = time.time() - t0
evr, cum = pca.explained_variance_ratio_, np.cumsum(pca.explained_variance_ratio_)
K90 = int(np.searchsorted(cum, 0.90) + 1)
print(f"PCA runtime: {t_pca:.3f} s | components needed for 90 % of the variance: {K90}  (cumulative = {cum[K90-1]:.3f}; with {K90-1}: {cum[K90-2]:.3f})")
print("Explained variance ratio:", np.round(evr, 3))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(range(1, len(evr) + 1), evr, color="steelblue", label="individual"); ax.plot(range(1, len(evr) + 1), cum, "-o", color="crimson", label="cumulative")
ax.axhline(.9, color="k", ls="--", lw=1); ax.axvline(K90, color="k", ls=":", lw=1); ax.text(K90 + .1, .5, f"{K90} components → 90 %", fontsize=9)
ax.set_xlabel("principal component"); ax.set_ylabel("explained variance ratio"); ax.set_xticks(range(1, len(evr) + 1)); ax.legend(); ax.set_title("PCA scree plot")
plt.tight_layout(); savefig("q6_2_scree"); plt.show()

def three_panel(emb, title, fname, s=2, alpha=.5, xlabel="dim 1", ylabel="dim 2"):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
    for ax, (col, cmap, lab) in zip(axes, [(hour, "twilight_shifted", "pickup hour"), (weekend, "coolwarm", "weekend (1) / weekday (0)"), (dist_col, "viridis", "trip distance (km, clipped at 99th pct)")]):
        o = np.random.default_rng(SEED).permutation(len(emb))
        sc = ax.scatter(emb[o, 0], emb[o, 1], c=col[o], cmap=cmap, s=s, alpha=alpha, rasterized=True, linewidths=0)
        plt.colorbar(sc, ax=ax, label=lab); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(lab, fontsize=10)
    plt.suptitle(title, fontweight="bold"); plt.tight_layout(); savefig(fname); plt.show()
three_panel(Z[:, :2], f"PCA – first two components ({evr[0]+evr[1]:.1%} of the variance)", "q6_2_pca_scatter", xlabel=f"PC1 ({evr[0]:.1%})", ylabel=f"PC2 ({evr[1]:.1%})")

load = pd.DataFrame(pca.components_[:4].T, index=FEATURES, columns=[f"PC{i}" for i in range(1, 5)])
fig, ax = plt.subplots(figsize=(6, 5)); sns.heatmap(load, cmap="RdBu_r", center=0, annot=True, fmt=".2f", ax=ax); ax.set_title("PCA loadings (first four components)")
plt.tight_layout(); savefig("q6_2_loadings"); plt.show()
print(load.round(2).to_string())
for pc in ("PC1", "PC2", "PC3"):
    print(pc, "largest |loadings|:", load[pc].abs().sort_values(ascending=False).head(4).index.tolist())

**Variance explained:** the variance is spread out over many components: PC1 = 22.1 %, PC2 = 13.9 %, PC3 = 12.3 %, PC4 = 10.2 %, … – the first two components capture only **36.0 %**. **8 components are needed to explain 90 %** of the variance
(7 give 83.7 %, 8 give 90.2 %; the 11th component has ≈ 0 variance because the 11 columns contain an exact dependency – speed = distance/duration is a linear combination of the logged columns: `log speed = log distance − log duration + const`).
So the data are genuinely high-dimensional; a 2-D PCA picture cannot be a faithful summary.

**Loadings and interpretation** (heat-map and table above):
* **PC1 (22 %) – trip length / "long fast trips out of the core".** Largest loadings: `log_distance` (+0.60), `log_duration` (+0.46), `log_speed` (+0.36), `pickup_lon` (+0.35), `dropoff_lon` (+0.28). High PC1 = long, long-lasting, fast trips that end/start to the east (towards Queens and JFK/LaGuardia);
  the right-hand tail of the scatter plot (PC1 > 4) consists of the yellow, 15–20 km trips – the airport runs. Low PC1 = short Manhattan hops.
* **PC2 (14 %) – uptown/downtown (north-east) position.** Dominated by `dropoff_lat` (+0.61), `pickup_lat` (+0.53), `dropoff_lon` (+0.43): high PC2 = trips in the northern/eastern part of Manhattan, low PC2 = downtown/west side – i.e. the long diagonal axis of the island.
* **PC3 (12 %) – time of day and day of week.** `hour_cos` (+0.61), `log_speed` (+0.48), `log_duration` (−0.44), `dow_sin` (−0.37): night-time (hour_cos high around midnight) trips are fast and short vs afternoon trips that are slow – the congestion pattern of Section 3.5.
* **PC4 (10 %)** is almost purely `hour_sin` (+0.78): the morning ↔ evening contrast.

**Colour views of PC1–PC2:**
* **Distance (right):** the clearest structure – distance increases to the right along PC1 and the long trips (yellow) form **separate small clouds at PC1 ≈ 2.5–3 and PC1 ≈ 5–6**, which are the airport groups (two airports/directions); this is the only part of the PCA plot with visible separate groups.
* **Hour (left):** the colour is a mixture – hour is **not** captured by PC1/PC2 (its loadings are on PC3/PC4), so the points of different hours overlap almost entirely.
* **Weekend vs weekday (middle):** the two classes are fully overlapped in PC1–PC2, a weekend–weekday difference is again in the later components.
* **Overall:** PCA with two components separates *long (airport) vs short trips* and *where in the city*, but not *when*, and the bulk of the trips forms one dense unstructured cloud.

### 6.3 t-SNE [3 marks]

t-SNE (`init="pca"`, `random_state=42`, default learning rate) with perplexities **5, 30 and 100** on the same standardised matrix; each embedding is shown coloured by hour, weekday/weekend and distance.
Perplexity is the effective number of neighbours each point "pays attention to" when the local similarities are defined.

In [ ]:
# ---- 6.3 t-SNE: three perplexities ---------------------------------------------------------------------------------
PERPLEXITIES = [5, 30, 100]
def run_tsne(perp, seed=SEED):
    t0 = time.time()
    emb = TSNE(n_components=2, perplexity=perp, init="pca", random_state=seed, n_jobs=-1).fit_transform(X)
    return emb, time.time() - t0
tsne = {}
for perp in PERPLEXITIES:
    tsne[perp] = cached(f"tsne_p{perp}_s{SEED}", lambda perp=perp: run_tsne(perp))
    print(f"t-SNE perplexity {perp:>3}: {tsne[perp][1]:.0f} s")

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 16))
for r, perp in enumerate(PERPLEXITIES):
    emb = tsne[perp][0]; o = np.random.default_rng(SEED).permutation(len(emb))
    for ax, (col, cmap, lab) in zip(axes[r], [(hour, "twilight_shifted", "pickup hour"), (weekend, "coolwarm", "weekend (1) / weekday (0)"), (dist_col, "viridis", "trip distance (km)")]):
        sc = ax.scatter(emb[o, 0], emb[o, 1], c=col[o], cmap=cmap, s=1.5, alpha=.5, rasterized=True, linewidths=0)
        ax.set_title(f"perplexity {perp} – colour: {lab}", fontsize=10); ax.set_xticks([]); ax.set_yticks([]); plt.colorbar(sc, ax=ax, shrink=.8)
plt.tight_layout(); savefig("q6_3_tsne_grid"); plt.show()

**Effect of perplexity** (3 × 3 grid above):
* **Perplexity 5 (57 s)** – very local: the picture is a single, almost circular blob in which neighbouring points are locally consistent (hour and weekend colours vary smoothly) but **no clear groups** appear; the weekend half (red) is a contiguous wedge,
  and the long trips (yellow) form a few small chains at the right edge, fragmented into many tiny pieces. Small perplexities over-emphasise noise and cannot see the larger structure.
* **Perplexity 30 (69 s)** – the most informative: a **flower-like structure of ≈ 7 large lobes** plus a few small separate islands. Colouring by weekend shows that **two lobes are weekend (red) and five are weekday (blue)**: each lobe is one day of the week
  (because the day-of-week encoding takes seven discrete values, see 6.1), and *inside* each lobe the colour changes smoothly with the hour (blue morning → red afternoon → purple night). The small separate islands on the right are the **long trips (yellow/green in the distance plot)**, i.e. the airport runs.
* **Perplexity 100 (110 s)** – the same groups, merged more cleanly: **fewer, larger and more compact lobes**, better separated from each other and with clearer hour gradients inside, but small structures (tiny islands) begin to be absorbed;
  it also costs more time (the cost grows with perplexity). The relative positions of the lobes also change between the perplexities, which shows that *distances between islands are not stable*, only membership of the islands is.
* **Summary:** increasing perplexity moves the picture from "local noise" (5) through "clusters" (30) to "larger, smoother, more global-looking clusters" (100). Perplexity 30 resolved both the 7 daily lobes and the small airport islands best (see the trustworthiness scores in 6.5).

### 6.4 UMAP [3 marks]

UMAP (`random_state=42`) is run for all combinations of `n_neighbors` ∈ {5, 15, 50} and `min_dist` ∈ {0.0, 0.5}; the same three colourings are shown (three figures: hour, weekend, distance).
`n_neighbors` controls how local the neighbourhood graph is (small = local detail, large = global structure) and `min_dist` how tightly points may be packed in the low-dimensional picture (0 = clumped, 0.5 = spread out).

In [ ]:
# ---- 6.4 UMAP: n_neighbors x min_dist ----------------------------------------------------------------------------------
NEIGHBORS, MIN_DISTS = [5, 15, 50], [0.0, 0.5]
def run_umap(nn, md_, seed=SEED):
    t0 = time.time()
    emb = umap.UMAP(n_neighbors=nn, min_dist=md_, random_state=seed).fit_transform(X)
    return emb, time.time() - t0
umaps = {}
for nn in NEIGHBORS:
    for md_ in MIN_DISTS:
        umaps[(nn, md_)] = cached(f"umap_n{nn}_d{md_}_s{SEED}", lambda nn=nn, md_=md_: run_umap(nn, md_))
        print(f"UMAP n_neighbors={nn:>2}, min_dist={md_}: {umaps[(nn, md_)][1]:.0f} s")

In [ ]:
for (col, cmap, lab, fname) in [(hour, "twilight_shifted", "pickup hour", "hour"), (weekend, "coolwarm", "weekend (1) / weekday (0)", "weekend"), (dist_col, "viridis", "trip distance (km)", "distance")]:
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    for j, nn in enumerate(NEIGHBORS):
        for i, md_ in enumerate(MIN_DISTS):
            emb = umaps[(nn, md_)][0]; o = np.random.default_rng(SEED).permutation(len(emb)); ax = axes[i, j]
            sc = ax.scatter(emb[o, 0], emb[o, 1], c=col[o], cmap=cmap, s=1.5, alpha=.5, rasterized=True, linewidths=0)
            ax.set_title(f"n_neighbors = {nn}, min_dist = {md_}", fontsize=10); ax.set_xticks([]); ax.set_yticks([]); plt.colorbar(sc, ax=ax, shrink=.8)
    plt.suptitle(f"UMAP – coloured by {lab}", fontweight="bold"); plt.tight_layout(); savefig(f"q6_4_umap_{fname}"); plt.show()

**Effect of the parameters:**
* **`n_neighbors`.** With **5** the embedding is the most fragmented: the 7 daily groups appear, but each is stretched, a number of small satellite fragments appear (over-segmentation of noise), and the groups' internal hour gradient is rougher.
  With **15** the groups are compact and complete and the small airport islands are visible and separate. With **50** the daily groups are even more regular – similar sizes and shapes, arranged like a **flower with 7 petals** – the airport islands are still separate but the fine detail (small satellite
  fragments) is smoothed away. Larger `n_neighbors` therefore trades local detail for a cleaner, more global arrangement.
* **`min_dist`.** With **0.0** the points are packed into **very tight, clearly separated islands with white space between them** – ideal for seeing and clustering groups, but the internal structure is squeezed (density differences within a group are hidden, and the apparent gaps are exaggerated).
  With **0.5** the points are spread over the plane: the groups still exist (the hour gradient is clearly visible inside each petal) but they touch each other and the gaps between them shrink or disappear, making it harder to define cluster boundaries; it is better for seeing *internal* structure.
* **What stays the same:** in all six settings the main message is identical to PCA/t-SNE – seven day-of-week groups (two of them weekend, see the weekend figure) in which hour changes smoothly, plus a handful of small separate groups of long trips (distance figure).
  UMAP ran in 6–30 s per setting (the first 30 s includes numba compilation) vs 57–110 s for t-SNE.

### 6.5 Density-based clustering and cluster profiles [6 marks]

**Which embedding is "best"?** For t-SNE and UMAP the brief asks for the best 2-D embedding of the previous parts. We choose it **quantitatively**, with the *trustworthiness* score (how many of each point's 15 nearest neighbours in the embedding are also near neighbours in the original 11-D space;
1 = perfect, computed on a fixed random subset of 4 000 trips) and, as a second criterion, the Spearman correlation between pairwise distances in the original space and in the embedding (global structure). The table below lists all candidates.
The highest trustworthiness is used (t-SNE: perplexity 100 = 0.980, just ahead of perplexity 30 = 0.977; UMAP: `n_neighbors = 50, min_dist = 0.0` = 0.960).

**Clustering method and parameters.** DBSCAN with a common `min_samples = 30` (a cluster must hold at least 30 trips, i.e. 0.15 % of the sample, so that tiny accidental groups are treated as noise; the same value is used for all methods to keep them comparable).
`ε` is then chosen **separately for each space with a k-distance plot** (distance of every point to its 30th nearest neighbour, sorted): the curve is nearly flat (points inside dense regions) and then rises steeply (points in sparse regions/noise); we take the knee
(the point furthest from the chord of the curve, ignoring the most extreme 2 % of points) as ε: PCA 1.42, t-SNE 2.53, UMAP 0.16 – the units differ because the three spaces have different scales, which is why ε cannot be shared.
For PCA the clustering is run in the **8-dimensional** space that explains 90 % of the variance, and is *visualised* on the first two components; for t-SNE and UMAP it is run on the 2-D embedding itself.

In [ ]:
# ---- 6.5 clustering --------------------------------------------------------------------------------------------------
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

# which 2-D t-SNE / UMAP embedding is "best"?  Use trustworthiness (local neighbourhood preservation, k = 15) on a fixed random subset,
# and the Spearman correlation of pairwise distances (global structure) as a second criterion.
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist
sub_t = np.random.default_rng(SEED).choice(N_SAMPLE, 4000, replace=False)
sub_g = sub_t[:1500]
cands = {"PCA 2D": Z[:, :2]}
cands.update({f"t-SNE perp={p}": tsne[p][0] for p in PERPLEXITIES})
cands.update({f"UMAP nn={nn}, md={md_}": umaps[(nn, md_)][0] for nn in NEIGHBORS for md_ in MIN_DISTS})
d_hi = pdist(X[sub_g])
qual = []
for k, E in cands.items():
    qual.append({"embedding": k, "trustworthiness(k=15)": trustworthiness(X[sub_t], E[sub_t], n_neighbors=15),
                 "global dist. corr (Spearman)": spearmanr(d_hi, pdist(E[sub_g]))[0]})
qual = pd.DataFrame(qual).set_index("embedding").round(3)
BEST_TSNE = max(PERPLEXITIES, key=lambda p: qual.loc[f"t-SNE perp={p}", "trustworthiness(k=15)"])
BEST_UMAP = max(umaps, key=lambda k: qual.loc[f"UMAP nn={k[0]}, md={k[1]}", "trustworthiness(k=15)"])
print(qual.to_string()); print("\nBest t-SNE perplexity:", BEST_TSNE, "| best UMAP:", BEST_UMAP)
qual

In [ ]:
def kdist_eps(E, k):
    """Distance to the k-th nearest neighbour for every point (sorted) and the 'knee' (point of maximum distance from the chord) as eps."""
    nn = NearestNeighbors(n_neighbors=k).fit(E); d = np.sort(nn.kneighbors(E)[0][:, -1])
    x = np.linspace(0, 1, len(d)); y = (d - d.min()) / (d.max() - d.min())
    # the curve is flat then explodes (outliers) – look for the knee on the part below the 98th percentile of points
    m = int(len(d) * .98); xx, yy = x[:m] / x[m - 1], (d[:m] - d[0]) / (d[m - 1] - d[0])
    knee = int(np.argmax(xx - yy))              # largest gap between the diagonal chord and the curve
    return d, d[knee], knee / len(d)

MIN_SAMPLES = 30                                  # a cluster must hold at least 30 trips (0.15 % of the sample)
spaces = {"PCA (first %d PCs)" % K90: Z[:, :K90], f"t-SNE (perplexity {BEST_TSNE})": tsne[BEST_TSNE][0], f"UMAP (n_neighbors={BEST_UMAP[0]}, min_dist={BEST_UMAP[1]})": umaps[BEST_UMAP][0]}
kd = {k: kdist_eps(E, MIN_SAMPLES) for k, E in spaces.items()}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for ax, (k, (d, eps, frac)) in zip(axes, kd.items()):
    ax.plot(np.arange(len(d)) / len(d), d); ax.axhline(eps, color="crimson", ls="--"); ax.set_title(f"{k}\nk-distance (k={MIN_SAMPLES}); knee → ε = {eps:.3f}", fontsize=10)
    ax.set_xlabel("points sorted by k-distance (fraction)"); ax.set_ylabel(f"distance to {MIN_SAMPLES}th neighbour"); ax.set_ylim(0, np.percentile(d, 99.5))
plt.tight_layout(); savefig("q6_5_kdistance"); plt.show()
print({k: round(v[1], 4) for k, v in kd.items()})

In [ ]:
def cluster(E, eps, ms=MIN_SAMPLES): return DBSCAN(eps=eps, min_samples=ms).fit(E).labels_
labels = {k: cluster(E, kd[k][1]) for k, E in spaces.items()}
for k, lab in labels.items():
    sizes = np.bincount(lab[lab >= 0]) if lab.max() >= 0 else np.array([])
    print(f"{k:38s} eps={kd[k][1]:.3f}: {lab.max()+1:3d} clusters | noise {np.mean(lab == -1):6.1%} | largest cluster {sizes.max()/len(lab):.1%} | median size {np.median(sizes):.0f}" if len(sizes) else k + ": no clusters")

In [ ]:
# visualise the three clusterings in their own 2-D spaces (PCA: first two PCs)
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (k, lab) in zip(axes, labels.items()):
    E2 = Z[:, :2] if k.startswith("PCA") else spaces[k]
    noise = lab < 0; cm = plt.get_cmap("tab20")
    ax.scatter(E2[noise, 0], E2[noise, 1], s=1, c="#c8c8c8", alpha=.4, rasterized=True)
    ax.scatter(E2[~noise, 0], E2[~noise, 1], s=2, c=[cm(c % 20) for c in lab[~noise]], alpha=.7, rasterized=True)
    ax.set_title(f"{k}\n{lab.max()+1} clusters, {noise.mean():.1%} noise", fontsize=10); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); savefig("q6_5_clusters_2d"); plt.show()

In [ ]:
# ---- cluster profiles on the ORIGINAL (unscaled) features ----------------------------------------------------------
AIRPORTS = {"JFK": PLACES["JFK"], "Newark": PLACES["Newark"], "LaGuardia": (40.7769, -73.8740)}      # (lat, lon); LaGuardia added for the profile only
def near_airport(lon, lat, km=2.0):
    return np.any([haversine_np(lon, lat, lo, la) < km for la, lo in AIRPORTS.values()], axis=0)
S = sample.copy()
S["airport_trip"] = near_airport(S.pickup_longitude.values, S.pickup_latitude.values) | near_airport(S.dropoff_longitude.values, S.dropoff_latitude.values)
print(f"Trips with pickup or drop-off within 2 km of JFK, LaGuardia or Newark: {S.airport_trip.mean():.1%} of the sample")

def circ_hour(h):
    a = 2 * np.pi * np.asarray(h) / 24; s_, c_ = np.sin(a).mean(), np.cos(a).mean()
    return (np.arctan2(s_, c_) % (2 * np.pi)) * 24 / (2 * np.pi), np.hypot(s_, c_)

def time_label(h, R):
    if R < 0.35: return "all-day"
    if h >= 22 or h < 5: return "late-night"
    if h < 10: return "morning"
    if h < 16: return "midday"
    return "evening"

def name_cluster(r):
    if r["top day share"] > .6: wk = r["top day"]                                      # the cluster is (almost) one day of the week
    else: wk = "weekend" if r["weekend share"] > .45 else ("weekday" if r["weekend share"] < .15 else "mixed-week")
    ln = "very long" if r["median distance km"] > 12 else "long" if r["median distance km"] > 5 else "medium" if r["median distance km"] > 2 else "short"
    base = "airport runs" if r["airport share"] > .5 else "trips"
    return f"{time_label(r['mean hour'], r['hour concentration'])} {wk} {ln} {base}".strip()

def profile(lab):
    rows = []
    for c in [c for c in sorted(set(lab)) if c >= 0] + ([-1] if (lab < 0).any() else []):
        g = S[lab == c]; h, R = circ_hour(g.pickup_hour)
        lm, dm = nearest_landmark(g.pickup_latitude.median(), g.pickup_longitude.median(), max_km=1.5)
        rows.append({"cluster": c, "n": len(g), "% of sample": 100 * len(g) / len(S), "median distance km": g.distance_km.median(), "median duration min": g.duration_min.median(),
                     "median speed km/h": g.speed_kmh.median(), "mean hour": h, "hour concentration": R, "weekend share": g.is_weekend.mean(),
                     "airport share": g.airport_trip.mean(), "top day": g.day_name.astype(str).value_counts().index[0][:3], "top day share": g.day_name.astype(str).value_counts(normalize=True).iloc[0],
                     "pickups centred near": lm.replace("(no landmark < 1.5 km)", "—")})
    p = pd.DataFrame(rows).set_index("cluster")
    p["name"] = [name_cluster(r) if i >= 0 else "noise / unclustered" for i, r in p.iterrows()]
    return p.sort_values("n", ascending=False)

profiles = {k: profile(lab) for k, lab in labels.items()}
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
fmt = lambda p: p.round({"% of sample": 1, "median distance km": 2, "median duration min": 1, "median speed km/h": 1, "mean hour": 1, "hour concentration": 2, "weekend share": 2, "airport share": 2, "top day share": 2})
for k, p in profiles.items():
    print("\n=== " + k); print(fmt(p).to_string())

In [ ]:
# pickup locations of the biggest clusters, per method
WIN_LON, WIN_LAT = (-74.04, -73.76), (40.62, 40.86)
for k, lab in labels.items():
    p = profiles[k]; ids = [c for c in p.index if c >= 0]
    top = ids[:8] + [c for c in ids[8:] if p.loc[c, "airport share"] > .5][:4]          # the 8 largest clusters + up to 4 airport-dominated clusters
    ncol = 4; nrow = int(np.ceil(max(len(top), 1) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 6.2 * nrow)); axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, top):
        g = S[lab == c]
        ax.scatter(S.pickup_longitude, S.pickup_latitude, s=.3, c="#dddddd", rasterized=True)
        ax.scatter(g.pickup_longitude, g.pickup_latitude, s=3, c="crimson", alpha=.6, rasterized=True)
        ax.set_xlim(*WIN_LON); ax.set_ylim(*WIN_LAT); ax.set_aspect(1 / np.cos(np.radians(40.75))); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"cluster {c}: {p.loc[c, 'name']}\nn = {len(g):,} ({p.loc[c, '% of sample']:.1f} %)", fontsize=9)
    for ax in axes[len(top):]: ax.axis("off")
    plt.suptitle(f"Pickup locations of the 8 largest + airport clusters – {k}", fontweight="bold", y=1.0); plt.tight_layout()
    savefig("q6_5_maps_" + k.split(" ")[0].lower().replace("-", "")); plt.show()

**Numbers of clusters and noise** (printed above):

| Method (space clustered) | ε | clusters | noise | largest cluster |
|---|---|---|---|---|
| PCA, 8 PCs | 1.42 | **6** | **8.9 %** | 88.7 % |
| t-SNE (perplexity 100), 2-D | 2.53 | **19** | **2.6 %** | 29.1 % |
| UMAP (n_neighbors 50, min_dist 0), 2-D | 0.16 | **29** | **3.5 %** | 12.9 % |

**PCA.** The data in the PCA space form one big dense cloud. DBSCAN returns one giant cluster (88.7 % of the trips: "all-day, mixed-week, short trips" with median distance 1.95 km, duration 10.3 min, speed 12.2 km/h; pickups centred in Midtown/Grand Central)
plus a handful of tiny clusters; the *noise* group (8.9 %) is almost half airport trips (airport share 0.49, median distance 10.5 km), and the small clusters are airport runs of different times (midday weekday, morning weekday, evening weekend; 23–246 trips each).
PCA with density-based clustering therefore separates only **"airport/long trips" from "everything else"**; it cannot find the day-of-week/time-of-day structure, which is spread over many components.

**t-SNE (19 clusters).** The big weekday and weekend lobes are found, but DBSCAN merges lobes that touch: the largest cluster (29.1 %, "evening weekend short trips": 74 % weekend trips, Saturday 44 %) merges Saturday and Sunday (and some weekday trips)
and the second (26.4 %, "evening weekday short trips", no weekend trips) merges several weekdays (Monday is the largest share with 39 %). Single days are isolated for Thursday (two clusters: "midday Thu" and "evening Thu", each 100 % Thursday), Friday midday, Tuesday and Wednesday mornings, and Sunday late-night.
Six small clusters (0.2–2.3 % each) are airport runs.

**UMAP (29 clusters).** The tight islands (min_dist = 0) are separated cleanly and DBSCAN finds **each day of the week as its own cluster**: Friday, Saturday, Wednesday, Tuesday, Monday and Sunday are 8–13 % of the sample each and 99–100 % one day
(Thursday is split into a "midday" and an "evening" cluster); **Sunday is additionally split** into a "midday" cluster and a small late-night one (Saturday-night trips after midnight). Plus several small clusters of long trips.

**Cluster profiles** – the full tables above give, for every cluster, the median distance/duration/speed, the (circular) mean pickup hour and how concentrated it is (`hour concentration` 0 = spread over the day, 1 = one hour),
the weekend share, the airport share, the dominant weekday and where the pickups are centred. The names in the last column are generated by a **fixed rule from these statistics** (time of day from the circular mean hour; weekend/weekday/dominant day; length from the median distance: short < 2 km, medium 2–5, long 5–12, very long > 12;
"airport runs" if > 50 % of the trips start or end within 2 km of JFK, LaGuardia or Newark), so the names are reproducible and always consistent with the numbers. Summarising what the profiles show:

* **Day-of-week "baseline" trips (the great majority)** – short (median 1.7–2.0 km, 9–11 min, 10–14 km/h) Midtown/Grand Central/Times Square hops, spread over the whole day (hour concentration 0.22–0.30) for Monday, Tuesday, Wednesday, Friday and Saturday, e.g. UMAP clusters *all-day Fri/Sat/Wed/Tue/Mon short trips* (10–13 % each).
  Their speed ranks as 10.7 (Wednesday) – 12.7 (Saturday) km/h – the weekday congestion of Section 3.5.
* **Thursday split:** the *midday Thursday short trips* (mean hour 10.3, speed 10.1–10.2 km/h, the slowest cluster) and *evening Thursday short trips* (mean hour 20.2, speed 12.1 km/h) – the Thursday group is concentrated in time, in contrast with other days; the plot shows that
  its hours are distributed in two groups, with hour changing smoothly inside each.
* **Late-night weekend trips:** *late-night Sun short trips* (UMAP cluster 13, 1.9 %: mean hour 1.4, 99 % Sunday i.e. Saturday night, 1.8 km, 8.1 min, 13.7 km/h, pickups near Union Square) and, smaller, *late-night Sun long trips* (UMAP cluster 26, 0.4 %: 6.8 km, 18.2 min, 22.7 km/h, centred near Astor Place/NoHo) and
  *late-night weekend long trips* (cluster 17, 1.4 %: 6.8 km, 22.7 min, 18.2 km/h, mean hour 0.5, weekend share 0.62) – longer rides home after a night out (a likely, not verified, interpretation).
* **Fast long night rides:** *late-night mixed-week long trips* (UMAP cluster 0, 1.3 %: 9.7 km, 19.9 min, **29.3 km/h**, mean hour 23.4): fast late-evening/night cross-town or highway rides.
* **Airport runs (6.7 % of the sample are airport trips, and they are concentrated in a few small clusters):** the median distance ≈ 9.8 km, 28.5 min, 20.6 km/h for *LaGuardia-area* clusters (e.g. UMAP cluster 1, 2.3 %, pickups centred near LaGuardia, airport share 0.98)
  and **19.8–20.5 km, 40–43 min, 27–29 km/h for JFK** (clusters 7 and 12: JFK-centred pickups vs Times-Square-centred pickups – the two directions of the same route, to and from the airport). Their hours are spread over the whole day (hour concentration 0.3–0.4) and over the whole week.
  A further small cluster of ≈ 10 km runs (cluster 15, 1.6 %, pickups near Times Square, 78 % airport) is probably the Manhattan → LaGuardia direction.

*(Note: the cluster ids above refer to the UMAP table; the t-SNE table contains the corresponding airport clusters 1, 13, 6, 11, 14, 15.)*

**Pickup maps.** The maps (one panel per cluster, red = the cluster's pickups, grey = all sampled pickups) show that the day-of-week/time clusters are **spatially indistinguishable** – each covers essentially the whole of Manhattan with Midtown on top –
whereas the airport clusters (shown in the extra panels) differ spatially: their pickups are either at the airport itself or in Manhattan for the opposite direction. This confirms that location is not what separates the day-of-week clusters (they are separated by the day and time features); the only *spatially* defined clusters are the airport clusters.

### 6.6 Comparison of PCA, t-SNE and UMAP [4 marks]

The best embedding of each method (PCA first two components; t-SNE perplexity 100; UMAP `n_neighbors = 50, min_dist = 0`) is shown side by side, **top row coloured by pickup hour**, **bottom row highlighting the airport trips of Question 5**
(pickup or drop-off within 2 km of JFK, LaGuardia or Newark; 6.7 % of the sample). We then compare runtimes, the agreement of the three clusterings (adjusted Rand index), whether the clusters correspond to real structure in the original 11-D space,
and the sensitivity to the random seed (the best t-SNE with `init="random"` and the best UMAP are re-run with seeds 42, 1, 2 and re-clustered with the same procedure).
*Why random initialisation for the seed test:* with `init="pca"` t-SNE starts from a deterministic PCA layout, so different seeds give practically the same picture – the agreement would be perfect (and in an earlier run we indeed measured ARI = 1.000) for a trivial reason that says nothing about stability.

In [ ]:
# ---- 6.6 comparison ---------------------------------------------------------------------------------------------------
from sklearn.metrics import adjusted_rand_score
names = list(spaces)
best = {"PCA": Z[:, :2], "t-SNE": tsne[BEST_TSNE][0], "UMAP": umaps[BEST_UMAP][0]}
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for j, (m, E) in enumerate(best.items()):
    o = np.random.default_rng(SEED).permutation(len(E))
    sc = axes[0, j].scatter(E[o, 0], E[o, 1], c=hour[o], cmap="twilight_shifted", s=1.5, alpha=.5, rasterized=True, linewidths=0)
    axes[0, j].set_title(f"{m} – coloured by pickup hour"); plt.colorbar(sc, ax=axes[0, j], shrink=.8)
    a = S.airport_trip.values
    axes[1, j].scatter(E[~a, 0], E[~a, 1], s=1, c="#cccccc", alpha=.4, rasterized=True); axes[1, j].scatter(E[a, 0], E[a, 1], s=4, c="crimson", alpha=.8, rasterized=True)
    axes[1, j].set_title(f"{m} – airport trips (JFK/LGA/Newark within 2 km) in red")
    for r in (0, 1): axes[r, j].set_xticks([]); axes[r, j].set_yticks([])
plt.tight_layout(); savefig("q6_6_side_by_side"); plt.show()

# runtimes
rt = pd.Series({"PCA (all 11 PCs)": t_pca, **{f"t-SNE perplexity {p}": tsne[p][1] for p in PERPLEXITIES}, **{f"UMAP nn={k[0]}, md={k[1]}": umaps[k][1] for k in umaps}}).round(1)
print("Runtime (seconds, 20 000 trips, 4 CPU cores):"); print(rt.to_string())

In [ ]:
# agreement between the three clusterings (adjusted Rand index)
L = list(labels.values()); short = ["PCA", "t-SNE", "UMAP"]
ari_all = pd.DataFrame(index=short, columns=short, dtype=float); ari_core = ari_all.copy()
for i in range(3):
    for j in range(3):
        ari_all.iloc[i, j] = adjusted_rand_score(L[i], L[j])                                  # noise treated as one extra cluster
        m = (L[i] >= 0) & (L[j] >= 0)
        ari_core.iloc[i, j] = adjusted_rand_score(L[i][m], L[j][m]) if m.sum() > 1 else np.nan  # only trips clustered by both methods
print("ARI – noise counted as its own cluster"); print(ari_all.round(3).to_string())
print("\nARI – restricted to trips that BOTH methods put in a cluster (n shown below)")
print(ari_core.round(3).to_string())
print({f"{short[i]}&{short[j]}": int(((L[i] >= 0) & (L[j] >= 0)).sum()) for i in range(3) for j in range(i + 1, 3)})
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(ari_all.astype(float), annot=True, fmt=".2f", vmin=0, vmax=1, cmap="Blues", ax=axes[0]); axes[0].set_title("ARI (noise = extra cluster)")
sns.heatmap(ari_core.astype(float), annot=True, fmt=".2f", vmin=0, vmax=1, cmap="Blues", ax=axes[1]); axes[1].set_title("ARI (both-clustered trips only)")
plt.tight_layout(); savefig("q6_6_ari"); plt.show()

In [ ]:
# can the clusters be trusted?  (1) do they exist in the ORIGINAL 11-D space: silhouette + how well a kNN classifier on the original features recovers the labels
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
rows = []
for k, lab in labels.items():
    m = lab >= 0
    idx = np.where(m)[0]
    if len(set(lab[m])) < 2: rows.append({"method": k}); continue
    sel = np.random.default_rng(SEED).choice(idx, min(5000, len(idx)), replace=False)
    sil = silhouette_score(X[sel], lab[sel])
    acc = cross_val_score(KNeighborsClassifier(15), X[sel], lab[sel], cv=5).mean()
    base = np.bincount(lab[sel][lab[sel] >= 0]).max() / len(sel)
    rows.append({"method": k, "clusters": len(set(lab[m])), "silhouette in original X": sil, "kNN accuracy from original X": acc, "majority-class baseline": base,
                 "airport trips in best cluster (% of all airport trips)": 100 * np.bincount(lab[S.airport_trip.values & m]).max() / max(1, S.airport_trip.sum()) if (S.airport_trip.values & m).any() else 0})
trust = pd.DataFrame(rows).set_index("method").round(3); print(trust.to_string()); trust

In [ ]:
# sensitivity to the random seed.
# NB: with init="pca" t-SNE starts from a deterministic PCA layout, so changing the seed hardly changes the result (ARI would be ~1 trivially).
# A fair test of seed sensitivity therefore uses init="random" for t-SNE; UMAP is run with different seeds directly.
def run_tsne_random(perp, seed):
    t0 = time.time(); emb = TSNE(n_components=2, perplexity=perp, init="random", random_state=seed, n_jobs=-1).fit_transform(X); return emb, time.time() - t0
SEEDS = [42, 1, 2]
seed_lab = {"t-SNE (random init)": [], "UMAP": []}; seed_emb = {"t-SNE (random init)": [], "UMAP": []}
for s_ in SEEDS:
    e_t = cached(f"tsne_rand_p{BEST_TSNE}_s{s_}", lambda s_=s_: run_tsne_random(BEST_TSNE, s_))
    e_u = umaps[BEST_UMAP] if s_ == SEED else cached(f"umap_n{BEST_UMAP[0]}_d{BEST_UMAP[1]}_s{s_}", lambda s_=s_: run_umap(*BEST_UMAP, seed=s_))
    for m, e in (("t-SNE (random init)", e_t), ("UMAP", e_u)):
        seed_emb[m].append(e[0]); seed_lab[m].append(cluster(e[0], kdist_eps(e[0], MIN_SAMPLES)[1]))
    print(f"seed {s_}: t-SNE {e_t[1]:.0f}s, {seed_lab['t-SNE (random init)'][-1].max()+1} clusters, noise {np.mean(seed_lab['t-SNE (random init)'][-1]==-1):.1%} | "
          f"UMAP {e_u[1]:.0f}s, {seed_lab['UMAP'][-1].max()+1} clusters, noise {np.mean(seed_lab['UMAP'][-1]==-1):.1%}")
rows = []
for m, ls in seed_lab.items():
    rows.append({"method": m, "clusters per seed": [int(l.max() + 1) for l in ls], "ARI seeds 42-1": adjusted_rand_score(ls[0], ls[1]), "ARI seeds 42-2": adjusted_rand_score(ls[0], ls[2]), "ARI seeds 1-2": adjusted_rand_score(ls[1], ls[2])})
seed_tbl = pd.DataFrame(rows).set_index("method").round(3); print(seed_tbl.to_string())

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for r, m in enumerate(seed_emb):
    for c, s_ in enumerate(SEEDS):
        E = seed_emb[m][c]; axes[r, c].scatter(E[:, 0], E[:, 1], c=hour, cmap="twilight_shifted", s=1, alpha=.5, rasterized=True, linewidths=0)
        axes[r, c].set_title(f"{m}, seed {s_}", fontsize=10); axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
plt.tight_layout(); savefig("q6_6_seed_sensitivity"); plt.show()
seed_tbl

**Which groups of trips appear in all three methods?**
* **Airport trips** – the clearest consistent finding (bottom row). In **t-SNE** and **UMAP** the airport trips (red) form **several compact islands completely separated from the bulk of the 20 000 trips** (UMAP: three well separated airport islands plus smaller ones – JFK vs LaGuardia and the two directions, as the profiles in 6.5 suggest); in **PCA** they also lie away from the main cloud
  (to the right of PC1, which is the "long trip" direction) but as a **diffuse, partly overlapping band** rather than separated groups. So yes: the airport trips of Question 5 form their own group in all three methods, sharply in the non-linear ones and only loosely in PCA.
* **Day-of-week structure** – t-SNE and UMAP show seven day-of-week groups (each with a smooth hour gradient inside, top row), while PCA shows none: the seven groups come from the day-of-week features, which only contribute a small part of the variance in each linear component, so PCA (a global linear method) overlooks them.
  The consequence of our feature design (two of 11 features take only 7 values) is that these groups dominate the non-linear pictures; they are real in the sense that they exist in the input features, but they are *not* distinct "types of trip" in the sense of different locations, distances or speeds (maps in 6.5: all cover the same Manhattan footprint).
* **Late-night weekend trips** form separate small islands in the non-linear embeddings (6.5: *late-night Sun/weekend* clusters).

**What PCA reveals / misses compared with the non-linear methods.** PCA reveals the **dominant linear axes**, which are interpretable through the loadings (PC1 = trip length/speed, PC2 = uptown-downtown, PC3–PC4 = time of day), preserves global distances best
(Spearman correlation between original and embedded pairwise distances **0.75**, versus 0.41 for the chosen t-SNE and 0.36 for the chosen UMAP) and runs in a fraction of a second. It misses the *local* structure: its trustworthiness is only **0.77** (versus 0.98 for t-SNE, 0.96 for UMAP) – a 2-D linear projection captures only 36 % of the variance –
and it cannot separate the day-of-week/time groups or the many small airport groups, which require non-linear neighbourhood relations.

**Local vs global structure, and what can be interpreted.**
* **Local structure:** t-SNE (trustworthiness 0.980) and UMAP (0.960 for the chosen setting; 0.918–0.960 over all six) preserve neighbourhoods far better than PCA (0.768). t-SNE is slightly better locally, and the choice of UMAP's `min_dist = 0` improves it (0.5 gives 0.92–0.93).
* **Global structure:** both non-linear methods preserve it poorly (distance correlation 0.2–0.4 for all t-SNE/UMAP settings, vs 0.75 for PCA). **The distances between clusters in t-SNE/UMAP should therefore not be interpreted**: the relative positions of the lobes change with perplexity (6.3) and with the seed
  (see the seed-sensitivity figure: the arrangement of the lobes/islands differs between seeds, and random-initialised t-SNE is visibly more fragmented than the PCA-initialised picture), and the white gaps between islands are largely created by the algorithm (the silhouette of the clusters in the original space is ≈ 0, see below, i.e. there is no empty gap in the data). **Cluster *sizes* are not reliable either:** t-SNE equalises the density of points
  (dense regions are expanded, sparse ones contracted) and UMAP with `min_dist = 0` packs everything tightly, so the area of an island does not measure its population or its spread in the original space.
  Only *membership* (which points are together) and, to a lesser degree, local neighbourhoods can be read.

**Runtime and sensitivity to hyper-parameters and seed** (20 000 trips, 4 cores):

| | runtime | hyper-parameter sensitivity | seed sensitivity |
|---|---|---|---|
| PCA | **0.16 s** (all components) | none (deterministic; only the number of components) | none (deterministic) |
| t-SNE | **57 s (perplexity 5), 69 s (30), 110 s (100)**; 93–116 s with random init | high: perplexity 5 gives no groups (trustworthiness 0.945), 30 and 100 are similar (0.977 / 0.980) but arrange the lobes differently | **moderate/high** – with random init the clusterings from three seeds have 22/24/21 clusters and ARI 0.85 (42 vs 1), 0.45 (42 vs 2), 0.39 (1 vs 2); with `init="pca"` the result is nearly seed-independent (ARI = 1.0) but that is only because of the deterministic initialisation |
| UMAP | **6–44 s** (the 30 s of the first call includes numba compilation) | high for the cluster count: `n_neighbors` 5/15/50 and `min_dist` 0/0.5 change fragmentation and gap sizes (6.4) | **lower** – clusters 29/20/23 for the three seeds, ARI 0.86, 0.86, 0.90 |

**How well do the clusters from the three methods agree?** Adjusted Rand index between the three clusterings (noise as its own group / only trips clustered by both):

| pair | ARI (noise = own class) | ARI (both clustered) |
|---|---|---|
| PCA – t-SNE | 0.083 | 0.027 |
| PCA – UMAP | 0.040 | 0.013 |
| **t-SNE – UMAP** | **0.420** | **0.449** |

**t-SNE and UMAP agree moderately** (ARI ≈ 0.42–0.45: they agree on the airport groups, the single-day groups and the late-night groups, but differ in where they merge neighbouring days – t-SNE merges Saturday/Sunday and several weekdays into big clusters, UMAP separates them),
while **PCA agrees almost not at all with either** (ARI 0.01–0.08): PCA-DBSCAN produced one giant cluster (88.7 %) and a few airport groups, so it cannot agree with the 19–29 day/time clusters of the non-linear methods. ARI is also depressed because the cluster *numbers* differ (6 vs 19 vs 29): ARI penalises splitting a group in two even when the split is consistent.

**Can the clusters found in t-SNE/UMAP be trusted as real groups in the original data?** Partly:
* **Yes, in terms of membership.** A k-nearest-neighbour classifier working *only on the original 11 standardised features* recovers the t-SNE and UMAP cluster labels with **95.9 % and 96.2 % cross-validated accuracy** (the majority-class baselines are 30.3 % and 13.4 %), i.e. the clusters are consistent with neighbourhoods in the original feature space –
  not an artefact of the optimisation. The airport clusters are additionally confirmed independently by Question 5 (their members are within 2 km of an airport).
* **No, in terms of number, size, separation and arrangement.** (i) the silhouette of the clusters, computed in the *original* space, is **−0.011 (t-SNE) and 0.000 (UMAP)** – i.e. no separation: there is no empty space between day-of-week groups in 11-D (they differ in only two of 11 coordinates);
  the gaps seen in the plots are produced by the embedding; (ii) the number of clusters changes with the seed and the parameters (t-SNE 21–24, UMAP 20–29, and 19/29 for the main run); (iii) cluster-merging depends on ε, which is read off from the embedding.
  The airport trips are split into several clusters (the largest one holds only 33 % (t-SNE) / 34 % (UMAP) of the airport trips), because direction/airport/time-of-day matter.
  **Conclusion:** clusters found in t-SNE or UMAP are hypotheses; they should be validated by profiling in the original space (as in 6.5), by repeating with different seeds/parameters, and by independent knowledge (here the airport definition from Q5).

**Recommendation.** For exploring this data set we would use **UMAP as the primary tool** (with `n_neighbors` ≈ 15–50 and a small `min_dist`), with **PCA alongside** and **t-SNE only as a cross-check**:
UMAP is several times faster than t-SNE, was more stable across seeds in our test (ARI 0.86–0.90 vs 0.39–0.85 for random-init t-SNE), preserves local structure almost as well (0.96 vs 0.98) and a comparable amount of global structure (distance correlation 0.36 vs 0.33–0.41 for t-SNE), and the clear islands it creates make density-based clustering straightforward.
PCA is indispensable as a first step because it is deterministic, instantaneous, tells us that 8 dimensions are needed to retain 90 % of the variance, and gives *interpretable axes*; and any cluster from UMAP/t-SNE must be checked with cluster profiles and maps in the original feature space.
*A caveat about our own result:* the seven day-of-week islands arise largely because day-of-week is a discrete feature in the feature matrix; encoding it as a weekend flag (or leaving it out) would remove those islands and put more emphasis on location, distance and speed, which would be a worthwhile follow-up.

### Summary of Section 6

| Part | Result |
|---|---|
| 6.1 | 11 features (4 coordinates, log distance/duration/speed, cyclic hour and day-of-week), standardised; 20 000 trips, seed 42. |
| 6.2 | PCA: 22.1 % + 13.9 % = 36 % in the first two components; **8 components for 90 %**; PC1 = trip length/speed (airport runs at the extreme), PC2 = uptown–downtown, PC3/PC4 = time of day. |
| 6.3 | t-SNE: perplexity 5 = noisy blob, 30 and 100 = seven day-of-week lobes with hour gradients + separate airport islands; 100 has the highest trustworthiness (0.980). |
| 6.4 | UMAP: larger `n_neighbors` → cleaner global arrangement, `min_dist` 0 → tight separated islands, 0.5 → touching groups; best (50, 0.0) with trustworthiness 0.960. |
| 6.5 | DBSCAN (min_samples 30, ε from k-distance): **PCA 6 clusters / 8.9 % noise; t-SNE 19 / 2.6 %; UMAP 29 / 3.5 %**; clusters are day-of-week/time groups plus airport and late-night groups; profiles and maps given. |
| 6.6 | Airport trips separate in all three methods (loosely in PCA); ARI t-SNE–UMAP 0.42, PCA with either ≈ 0.04–0.08; clusters are recoverable from the original features (≈ 96 % kNN accuracy) but not separated there (silhouette ≈ 0); UMAP recommended, PCA for interpretation. |

---
## Reproducibility notes
* Random seed 42 everywhere (sample, t-SNE, UMAP); the seed-sensitivity test in 6.6 deliberately uses seeds 1 and 2 as well.
* `train.csv` is read from the repository's `nyc-taxi-trip-duration.zip` if it is not extracted; the t-SNE/UMAP embeddings are cached in `cache/` (not committed – delete to recompute; a full from-scratch run takes about 15–20 minutes on 4 cores).
* Figures used in the PDF are saved in `figures/` and the interactive hotspot maps (Section 4.2) in `maps/`.
* Packages: pandas, numpy, matplotlib, seaborn, scipy, scikit-learn, umap-learn, folium (all installable with pip).